<a href="https://colab.research.google.com/github/binhhuongvu/Landslide-Dectection/blob/main/Final_Systems_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Geospatial Foundation Models for Landslide Detection 🔎⛰️

**Goal:** Compare three segmentation architectures on the Landslide4Sense (L4S) benchmark— a frozen/LoRA-tuned Clay foundation model (Arch 1), a hybrid U-Net + Clay model (Arch 2), and a competition-style U-Net baseline — to determine whether a Geospatial Foundational Model improves pixel-level landslide detection over a standard methods.

**Evaluation metric:** Pixel-wise F1 score (matching competition protocol).

**Key techniques:** LoRA fine-tuning, two-stage training, Lovász–BCE loss, MC Dropout uncertainty, Grad-CAM interpretability.

# Part 1: Set Up
This section contains all components shared/accessed by Part 2,3,4. It will always have to be ran first. Then, any other part and independently run next.

## Install all project dependencies



In [ ]:
!pip install -q git+https://github.com/Clay-foundation/model.git  # Clay package
!pip install -q kagglehub h5py einops pyyaml
!pip install -q tqdm
!pip install -q peft
!pip uninstall -y torchao

import os, time, json, warnings, copy, requests, glob, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange
import h5py
import yaml
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model

from huggingface_hub import hf_hub_download, snapshot_download
from claymodel.module import ClayMAEModule
from claymodel.finetune.segment.factory import Segmentor

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
print('All dependencies installed and verified.')

## Download Landslide4Sense Dataset
Using `harshinde/LandSlide4Sense` on HuggingFace: the only publicly available version that includes **test set masks** (competition never released these). Contains all 3 splits fully labeled with 14-band `.h5` chips (handles multi-bandata).

In [ ]:
# ============================================================
# Dataset setup: one-time download + tar, fast untar each runtime
# ============================================================
import subprocess, shutil
from google.colab import drive
drive.mount('/content/drive') # Mount Gdrive into Colab for read/write

DRIVE_DIR  = '/content/drive/MyDrive/EPS210'
CKPT_DIR   = f'{DRIVE_DIR}/checkpoints'
#Location on Colab's fast disk
DATA_ROOT = '/content/landslide4sense'
os.makedirs(CKPT_DIR, exist_ok=True)

TAR_PATH = f'{DRIVE_DIR}/l4s.tar' # Compressed archive of dataset on Drive

if not os.path.exists(TAR_PATH):
    # First time ever: download + pack into tar
    print('First-time setup: downloading from HuggingFace (~9 GB)...')
    tmp = '/content/l4s_tmp' # into temp. folder
    snapshot_download(
        repo_id='harshinde/LandSlide4Sense',
        repo_type='dataset',
        local_dir=tmp,
    )
    print('Packing into tar on Drive (do this once)...')
    t0 = time.time()
    # Compress dataset into .tar file in Drive
    subprocess.run(['tar', '-cf', TAR_PATH, '-C', tmp, '.'], check=True)
    # Delete temp file (free up Colab disk space)
    shutil.rmtree(tmp)
    print(f'Tar saved → {TAR_PATH}  ({time.time()-t0:.1f}s)')
else:
    print(f'Tar found on Drive: {TAR_PATH}')

# Every runtime: fast untar to local SSD
if not os.path.exists(os.path.join(DATA_ROOT, 'images')):
    t0 = time.time()
    print('Extracting to local SSD...')
    os.makedirs(DATA_ROOT, exist_ok=True)
    subprocess.run(['tar', '-xf', TAR_PATH, '-C', DATA_ROOT], check=True)
    print(f'Ready in {time.time()-t0:.1f}s')
else:
    print(f'Local data ready: {DATA_ROOT}')
# List diles in dataset directory
print(f'Contents: {os.listdir(DATA_ROOT)}')

In [ ]:

# ============================================================
# Locate paired image/mask .h5 files for each split
# images/{split}/image_N.h5  ↔  annotations/{split}/mask_N.h5
# ============================================================
def find_paired_splits(root):
    splits = {} # Dictionary for each dataset split
    for split in ['train', 'validation', 'test']:
        # Construct paths (root/images/train or root/annos/train)
        img_dir = os.path.join(root, 'images', split)
        msk_dir = os.path.join(root, 'annotations', split)
        # Check there are matching splits
        if not os.path.isdir(img_dir) or not os.path.isdir(msk_dir):
            raise RuntimeError(f'Missing directories for split={split}\n'
                               f'  Expected: {img_dir}\n  and: {msk_dir}')
        # Finds all .5 images in split + sorts
        img_files = sorted(glob.glob(os.path.join(img_dir, '*.h5')))
        # Build mask lookup tabel: create dict (key: 12.h5, value: mask_12.h5)
        msk_map   = {
            os.path.basename(p).replace('mask_', ''): p
            for p in glob.glob(os.path.join(msk_dir, '*.h5'))
        }
        # For each image in sorted image list:
        # 1. remove 'image_' from filename, finds matching match key in msk_map
        pairs = [
            # 2. Create (image_path, mask_path tuple)
            (ip, msk_map[os.path.basename(ip).replace('image_', '')])
            for ip in img_files
            if os.path.basename(ip).replace('image_', '') in msk_map
        ]

        if not pairs:
            raise RuntimeError(f'No paired files found for split={split}')
        # Saves all pairs under 'split' key in splires dictionary
        # {'train': [(img1_path,mask1_path),...], 'val'}
        splits[split] = pairs

    return splits

SPLITS = find_paired_splits(DATA_ROOT) # Apply splits function to DATA_ROOT
# For each 'split' in SPLITS dict, prints number of pairs
for split, pairs in SPLITS.items():
    ex_img, ex_msk = pairs[0]
    print(f'{split:10s}: {len(pairs):4d} chips — '
          # Prints first img/mask pair example in 'split'
          f'img={os.path.basename(ex_img)} | mask={os.path.basename(ex_msk)}')

In [ ]:
# ============================================================
# Inspect one sample: confirm shape, dtype, band count
# ============================================================
BAND_NAMES = [
    'B1','B2','B3','B4','B5','B6','B7','B8','B9','B10','B11','B12',
    'Slope','DEM'
]

# Load one img + msk
def load_pair(img_path, msk_path):
    with h5py.File(img_path, 'r') as f:
        img = f['img'][:].astype(np.float32)   # (128, 128, 14)
    with h5py.File(msk_path, 'r') as f:
        key = list(f.keys())[0]                # handle 'mask' or 'gt' etc
        msk = f[key][:].astype(np.int64)       # (128, 128)
    return img, msk # return pairs as grid

sample_img, sample_mask = load_pair(*SPLITS['train'][0]) # Loads first training pair
print(f'img  shape : {sample_img.shape}   dtype: {sample_img.dtype}')
print(f'mask shape : {sample_mask.shape}  dtype: {sample_mask.dtype}')
print(f'mask values: {np.unique(sample_mask)}') # Check for binary values
print('\nPer-band range (one sample):')
for i, name in enumerate(BAND_NAMES): # Loop through each chan/band
    ch = sample_img[:, :, i]
    print(f'  {name:6s}  min={ch.min():8.3f}  max={ch.max():8.3f}  mean={ch.mean():8.3f}')

**Takeaways:** Not on the same scale as bands trained by Clay (from 0-10000). This could be because data has been preprocessed/scaled/cipping but still are not normalized (mean != 0)

In [ ]:
# ============================================================
# @title Class balance
# First-place paper: positive pixels = 2% of all training pixels
# ============================================================
pos_ratios  = [] # Fraction of positive pixel per img/chip
chips_w_pos = 0 # Count how many images contain any positive pixels

# Loop through all (img,msk) pair in training
for img_p, msk_p in tqdm(SPLITS['train'], desc='Scanning train chips'):
    with h5py.File(msk_p, 'r') as f:
        key = list(f.keys())[0]
        msk = f[key][:] # Open msk file
    r = msk.mean() # Fraction of pixels labeled 1 PER image/chip
    pos_ratios.append(r) # Add ratio to array
    if r > 0: chips_w_pos += 1

pos_ratios = np.array(pos_ratios)
print('=== Positive instance distribution (train split) ===')
print(f'Total chips (train) : {len(pos_ratios)}') # Number of training samples
print(f'Chips with any pos  : {chips_w_pos} ({100*chips_w_pos/len(pos_ratios):.1f}%)')
print(f'Overall pos ratio   : {pos_ratios.mean():.4f}  (paper reports ~0.02)')
# Average fraction of positive pixels across dataset
print(f'Chips > 10% pos     : {(pos_ratios > 0.10).sum()}')
print(f'Chips > 1%  pos     : {(pos_ratios > 0.01).sum()}')
print(f'Chips < 0.1% pos    : {(pos_ratios < 0.001).sum()} (near-empty chips)')

# Distribution of chips with any pos chips only
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pos_ratios[pos_ratios > 0], bins=50, color='#e63946', edgecolor='white', linewidth=0.3)
ax.axvline(0.01, color='orange', linestyle='--', linewidth=1.5, label='>1%') # Min useful signal
ax.axvline(0.10, color='red',    linestyle='--', linewidth=1.5, label='>10%') # Strong signal
ax.set_xlabel('Positive pixel fraction per chip', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Positive Instance Distribution (train)', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
# Save to Drive
plt.savefig(f'{DRIVE_DIR}/l4s_pos_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

EDA: understand why certain satellite bands (like SWIR) are more effective than others for identifying wildfire damage.

This step acts as a sanity check. If the separability index is low across all bands, no amount of complex architecture (like a Transformer or CNN) will perform well because the input data lacks a clear discriminative signal.

In [ ]:
# ============================================================
# @title Landslide4Sense feature validation / EDA
# ============================================================
subset_size = 20
sample_idx = 0
min_pixels_per_class = 50
split_name = "train"

pairs = SPLITS[split_name]
print(f"Using {split_name} split with {len(pairs)} samples")

band_names = BAND_NAMES if "BAND_NAMES" in globals() else [f"Band {i}" for i in range(14)]

def resolve_x_axis(names):
    wl = globals().get("L4S_WAVELENGTH", None)
    if isinstance(wl, dict):
        vals = np.array([wl.get(n, np.nan) for n in names], dtype=float)
        if np.isfinite(vals).all():
            return vals, "Wavelength (um)"
    try:
        vals = np.array(wl, dtype=float)
        if vals.ndim == 1 and len(vals) == len(names):
            return vals, "Wavelength (um)"
    except:
        pass
    return np.arange(len(names)), "Band Index"

def robust_normalize(x, pmin=2, pmax=98):
    x = np.asarray(x, dtype=np.float32)
    m = np.isfinite(x)
    if not m.any():
        return np.zeros_like(x)
    lo, hi = np.percentile(x[m], [pmin, pmax])
    if np.isclose(lo, hi):
        lo, hi = x[m].min(), x[m].max()
    if np.isclose(lo, hi):
        return np.zeros_like(x)
    return np.clip((x - lo) / (hi - lo), 0, 1)

def find_band(cands, names):
    names_l = [str(x).lower() for x in names]
    for c in map(str.lower, cands):
        for i, n in enumerate(names_l):
            if c == n or c in n:
                return i
    return None

x_axis, x_label = resolve_x_axis(band_names)
red_idx, green_idx, blue_idx, nir_idx = (
    find_band(["B4", "red"], band_names),
    find_band(["B3", "green"], band_names),
    find_band(["B2", "blue"], band_names),
    find_band(["B8", "nir"], band_names),
)

# -----------------------------
# Visualize one sample
# -----------------------------
img_i, mask_i = load_pair(*pairs[sample_idx])

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.ravel()
plot_i = 0

if None not in (red_idx, green_idx, blue_idx):
    rgb = np.stack([
        robust_normalize(img_i[..., red_idx]),
        robust_normalize(img_i[..., green_idx]),
        robust_normalize(img_i[..., blue_idx]),
    ], axis=-1)
    axes[plot_i].imshow(rgb)
    axes[plot_i].set_title(f"RGB ({band_names[red_idx]}, {band_names[green_idx]}, {band_names[blue_idx]})")
else:
    axes[plot_i].text(0.5, 0.5, "RGB bands not found", ha="center", va="center")
axes[plot_i].axis("off")
plot_i += 1

if None not in (nir_idx, red_idx, green_idx):
    false_color = np.stack([
        robust_normalize(img_i[..., nir_idx]),
        robust_normalize(img_i[..., red_idx]),
        robust_normalize(img_i[..., green_idx]),
    ], axis=-1)
    axes[plot_i].imshow(false_color)
    axes[plot_i].set_title(f"False Color ({band_names[nir_idx]}, {band_names[red_idx]}, {band_names[green_idx]})")
else:
    axes[plot_i].text(0.5, 0.5, "False-color bands not found", ha="center", va="center")
axes[plot_i].axis("off")
plot_i += 1

axes[plot_i].imshow(mask_i, cmap="Reds", vmin=0, vmax=1)
axes[plot_i].set_title("Mask")
axes[plot_i].axis("off")
plot_i += 1

axes[plot_i].axis("off")
plot_i += 1

for b in range(img_i.shape[-1]):
    if plot_i >= len(axes):
        break
    ax = axes[plot_i]
    band = img_i[..., b]
    m = np.isfinite(band)
    if m.any():
        vmin, vmax = np.percentile(band[m], [2, 98])
        if np.isclose(vmin, vmax):
            vmin, vmax = band[m].min(), band[m].max()
    else:
        vmin, vmax = 0, 1
    im = ax.imshow(band, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(band_names[b])
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plot_i += 1

for k in range(plot_i, len(axes)):
    axes[k].axis("off")

plt.suptitle("Sample chip visualization", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# -----------------------------
# Spectral signature: landslide vs background
# -----------------------------
landslide_spectra, background_spectra = [], []

for idx in range(min(subset_size, len(pairs))):
    img_i, mask_i = load_pair(*pairs[idx])
    valid = np.isfinite(img_i).all(axis=-1) & (mask_i >= 0)

    landslide_px = valid & (mask_i == 1)
    background_px = valid & (mask_i == 0)

    if landslide_px.sum() > min_pixels_per_class:
        landslide_spectra.append(img_i[landslide_px].mean(axis=0))
    if background_px.sum() > min_pixels_per_class:
        background_spectra.append(img_i[background_px].mean(axis=0))

print(f"Collected {len(landslide_spectra)} landslide spectra and {len(background_spectra)} background spectra")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if landslide_spectra:
    l_arr = np.array(landslide_spectra)
    l_mean, l_std = l_arr.mean(axis=0), l_arr.std(axis=0)
    axes[0].fill_between(x_axis, l_mean - l_std, l_mean + l_std, alpha=0.2, color="#e74c3c")
    axes[0].plot(x_axis, l_mean, "o-", color="#e74c3c", linewidth=2, label="Landslide")

if background_spectra:
    b_arr = np.array(background_spectra)
    b_mean, b_std = b_arr.mean(axis=0), b_arr.std(axis=0)
    axes[0].fill_between(x_axis, b_mean - b_std, b_mean + b_std, alpha=0.2, color="#2ecc71")
    axes[0].plot(x_axis, b_mean, "s-", color="#2ecc71", linewidth=2, label="Background")

axes[0].set_xlabel(x_label)
axes[0].set_ylabel("Mean feature value")
axes[0].set_title("(a) Mean spectral / feature signatures")
axes[0].legend()
axes[0].set_xticks(x_axis if x_label != "Band Index" else np.arange(len(band_names)))
axes[0].set_xticklabels(band_names, rotation=30, ha="right")

if landslide_spectra and background_spectra:
    separability = np.abs(l_arr.mean(axis=0) - b_arr.mean(axis=0)) / (
        np.sqrt(l_arr.std(axis=0) ** 2 + b_arr.std(axis=0) ** 2) + 1e-8
    )
    colors = ["#3498db" if s < 1 else "#e74c3c" for s in separability]
    axes[1].bar(band_names, separability, color=colors, edgecolor="black", alpha=0.8)
    axes[1].set_ylabel("Separability Index (d')")
    axes[1].set_title("(b) Landslide vs Background separability")
    axes[1].axhline(1.0, color="gray", linestyle="--", alpha=0.5)
    axes[1].tick_params(axis="x", rotation=30)

    print("\nTop bands by separability:")
    for rank, i in enumerate(np.argsort(separability)[::-1], 1):
        print(f"{rank:>2}. {band_names[i]:<12} d'={separability[i]:.3f}")
else:
    axes[1].text(0.5, 0.5, "Not enough valid samples", ha="center", va="center")
    axes[1].set_axis_off()

plt.suptitle("Feature Validation / EDA", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation**:
* Terrain features are weak (DEM + slope). But they could still add a lot of context information on a broader scale as auxilery information.
* B2,B3,B4,B5,B11,B12: extremely promising for model delineanation.
* B7,B8,B9,B10: very weak, might make model more unstable (also some of the bands that degrades attention model)

## Clay CheckPoint + New Sensor
The L4S dataset is **pre-scaled** (values ~1–6, not raw reflectance ~1000–2500).
Clay's YAML mean/std expect raw reflectance so they cannot be applied here.
Instead we register a new sensor and compute ALL 12 S2 band stats from training data,
following Clay's documentation. Wavelengths are physical constants — unchanged.

In [ ]:
# ============================================================
# !!!! If excuted previously, don't need to do it again.
# Download Clay v1.5 metadata file
# ! Assume: user already have checkpoint on Drive
# ============================================================
import yaml, requests # To read meta file

# Define paths in GDrive
CKPT_DIR      = f'{DRIVE_DIR}/checkpoints'
CKPT_PATH     = f'{CKPT_DIR}/clay-v1.5.ckpt'
METADATA_PATH = f'{CKPT_DIR}/metadata.yaml'
os.makedirs(CKPT_DIR, exist_ok=True) # Create checkpoint folder in GDrive if not exist

# Download metadata.yaml
if not os.path.exists(METADATA_PATH):
    r = requests.get('https://raw.githubusercontent.com/Clay-foundation/model/main/configs/metadata.yaml')
    open(METADATA_PATH, 'w').write(r.text)
    print('Downloaded metadata.yaml')
else:
    print('metadata.yaml already exists')

# Open YAML file from Drive and loads into Python dict.
with open(METADATA_PATH) as f:
    metadata = yaml.safe_load(f)

# Verify checkpoint file exists and size
size_gb = os.path.getsize(CKPT_PATH) / 1024**3
print(f'Clay checkpoint ready: {CKPT_PATH} ({size_gb:.1f} GB)')
print(f'Existing sensors in metadata: {list(metadata.keys())}')

*   If all pixels in a channel are invalid (NaN/Inf), then there is so useable data to contribute to statistics of that channel -> skip its stats for that band.
*   Loop over imgs = for each image: load img (H,W,C): for each channel: extract channel (H,W) -> find valid pixels -> if no valid, skip channel for this image only; else: flatten valid pixes, update running stats:

1.   count += number of valid pixels
2.   sum += sum of valid pixel values
3.   sumsq += sum of (value^2)










In [ ]:
# ============================================================
# TRAIN-ONLY normalization stats for all 14 channels
# Output:
#   TRAIN_NORM_STATS = {
#       's2': {'coastal': {'mean': ..., 'std': ...}, ...},
#       'terrain': {'slope': {'mean': ..., 'std': ...}, 'dem': {...}}
#   }
# - These TRAIN stats should be used to normalize train/val/test/inference
# - At load time, invalid pixels should be replaced with the band mean
#   BEFORE normalization, so they become ~0 after normalization
# ============================================================
import math

S2_BAND_INDICES  = list(range(12))    # Sentinel-2 channels: 0..11
TER_BAND_INDICES = [12, 13]           # Terrain channels: slope, DEM

L4S_SENSOR_NAME = 'landslide4sense-s2-12b'
# Human-readable band names in dataset channel order
L4S_BAND_ORDER = [
    'coastal', 'blue', 'green', 'red',
    'rededge1', 'rededge2', 'rededge3', 'nir',
    'water_vapor', 'cirrus', 'swir16', 'swir22'
]
TER_BAND_NAMES = ['slope', 'dem']
# Useulf for Clay metadata & model compatibility
L4S_WAVELENGTH  = {
    'coastal': 0.443, 'blue': 0.493, 'green': 0.560, 'red': 0.665,
    'rededge1': 0.704, 'rededge2': 0.740, 'rededge3': 0.783, 'nir': 0.842,
    'water_vapor': 0.945, 'cirrus': 1.375, 'swir16': 1.610, 'swir22': 2.190,
}

# Where to cache the train-only stats
TRAIN_STATS_PATH = f'{DRIVE_DIR}/train_norm_stats.json'


def compute_streaming_channel_stats(pairs, ch_indices, names, desc='Computing train stats'):
    """
    Compute per-channel mean/std over ALL valid pixels in a split, using
    streaming accumulation so memory stays small.
    """
    # One running accumulator per channel
    accum = {
        name: {'count': 0, 'sum': 0.0, 'sumsq': 0.0}
        for name in names
    }

    for img_p, _ in tqdm(pairs, desc=desc):
        # Load one image only -> memory efficient
        with h5py.File(img_p, 'r') as f:
            img = f['img'][:].astype(np.float32)   # expected shape: (128, 128, 14)

        # Process requested channels one by one
        for ch, name in zip(ch_indices, names):
            x = img[:, :, ch]

            # Keep only finite values for stats
            valid = np.isfinite(x)
            if not np.any(valid):
                # Entire channel invalid for this chip -> skip safely
                continue

            v = x[valid].astype(np.float64)  # float64 improves numerical stability

            accum[name]['count'] += v.size # number of valid pixels
            accum[name]['sum']   += float(v.sum()) # total values of valid pixels
            accum[name]['sumsq'] += float((v * v).sum())

    # Finalize mean/std per channel
    result = {}
    for name in names:
        c = accum[name]['count']

        if c == 0:
            raise RuntimeError(
                f'No valid pixels found for channel "{name}" while computing train stats.'
            )
        mean = accum[name]['sum'] / c
        var  = (accum[name]['sumsq'] / c) - (mean ** 2)

        # Numerical safety: tiny negative variance can occur from floating-point rounding
        var = max(var, 0.0)
        std = math.sqrt(var)

        # Prevent divide-by-zero later if a channel is constant
        std = max(std, 1e-6)

        result[name] = {
            'mean': float(mean),
            'std':  float(std),
            'count': int(c),
        }
    return result

# Compute TRAIN-ONLY stats, or load cached version if available
if os.path.exists(TRAIN_STATS_PATH):
    with open(TRAIN_STATS_PATH, 'r') as f:
        TRAIN_NORM_STATS = json.load(f)
    print(f'Loaded cached train-only normalization stats: {TRAIN_STATS_PATH}')
else:
    print('Computing TRAIN-ONLY normalization stats...')

    train_pairs = SPLITS['train']

    s2_stats = compute_streaming_channel_stats(
        pairs=train_pairs,
        ch_indices=S2_BAND_INDICES,
        names=L4S_BAND_ORDER,
        desc='Train S2 stats'
    )

    terrain_stats = compute_streaming_channel_stats(
        pairs=train_pairs,
        ch_indices=TER_BAND_INDICES,
        names=TER_BAND_NAMES,
        desc='Train terrain stats'
    )

    TRAIN_NORM_STATS = {
        's2': s2_stats,
        'terrain': terrain_stats,
    }

    with open(TRAIN_STATS_PATH, 'w') as f:
        json.dump(TRAIN_NORM_STATS, f, indent=2)

    print(f'Saved train-only normalization stats -> {TRAIN_STATS_PATH}')

# Print a readable summary table
print(f'\n{"Group":<10} {"Band":<12} {"Mean":>10} {"Std":>10} {"ValidCount":>14}')
print('-' * 62)
for band, s in TRAIN_NORM_STATS['s2'].items():
    print(f'{"S2":<10} {band:<12} {s["mean"]:>10.4f} {s["std"]:>10.4f} {s["count"]:>14}')
for band, s in TRAIN_NORM_STATS['terrain'].items():
    print(f'{"Terrain":<10} {band:<12} {s["mean"]:>10.4f} {s["std"]:>10.4f} {s["count"]:>14}')

In [ ]:
# ============================================================
# Register landslide4sense-s2-12b in metadata.yaml
# Uses TRAIN-ONLY normalization stats for the Sentinel-2 bands
# Wavelengths are physical constants, so they do not depend on
# preprocessing and are always safe to keep as fixed constants
# ============================================================
# Pull train-only Sentinel-2 stats from the new cached stats dict
train_s2 = TRAIN_NORM_STATS['s2']

# Add / overwrite the custom sensor entry in Clay metadata
metadata[L4S_SENSOR_NAME] = {
    'band_order':  L4S_BAND_ORDER,   # order of the 12 S2 channels
    'rgb_indices': [3, 2, 1],        # red, green, blue positions in band_order
    'gsd':         10.0,             # ground sampling distance in meters
    'bands': {
        # Per-band normalization stats computed from train split only
        'mean':       {b: train_s2[b]['mean'] for b in L4S_BAND_ORDER},
        'std':        {b: train_s2[b]['std']  for b in L4S_BAND_ORDER},

        # Physical wavelengths for Clay sensor registration
        'wavelength': {k: float(v) for k, v in L4S_WAVELENGTH.items()},
    },
}

# Save updated metadata back to disk so Clay can read the custom sensor entry
with open(METADATA_PATH, 'w') as f:
    yaml.safe_dump(metadata, f, sort_keys=False)

# Constants used later when building Clay inputs
# These are model-side constants, not normalization statistics
CLAY_WAVES  = torch.tensor([L4S_WAVELENGTH[b] for b in L4S_BAND_ORDER], dtype=torch.float32)
CLAY_GSD    = torch.tensor(10.0, dtype=torch.float32)
CLAY_TIME   = torch.tensor([1., 1., 1., 1.], dtype=torch.float32)
CLAY_LATLON = torch.tensor([0., 0., 0., 0.], dtype=torch.float32)

print(f'Registered: {L4S_SENSOR_NAME}')
print(f'Wavelengths: {CLAY_WAVES.shape} → expected 12 values for the model')

**Note:**
* Applying constant TIME and LATLON means the model is basing off of spectral-only instead of spectral + geo+ temporal data -> might lose performance compared to full input

## Dataset Augmentation

In [ ]:
# ============================================================
# Geometric aumentation
# ============================================================
def augment_geometric(chip, mask):
    """Random h/v flip + 90° rotation. chip (C,H,W), mask (H,W)."""
    # 50% chance horizontal flip
    if np.random.rand() > 0.5:
        chip = np.flip(chip, axis=2).copy(); mask = np.flip(mask, axis=1).copy()
    # 50% chance vert. flip
    if np.random.rand() > 0.5:
        chip = np.flip(chip, axis=1).copy(); mask = np.flip(mask, axis=0).copy()
    # Randomly rotate by 90dgrees increments
    k = np.random.randint(0, 4)
    if k > 0:
        chip = np.rot90(chip, k, axes=(1, 2)).copy()
        mask = np.rot90(mask, k, axes=(0, 1)).copy()
    return chip, mask

## Loss Functions

Three options implemented. Key insight: the competition evaluation metric is pixel-wise F1, so the loss should optimize toward overlap-based metrics, not just pixel-independent accuracy.

| Loss | Formula | When to use |
|------|---------|-------------|
| **Weighted BCE** | `BCE(logits, target, pos_weight=25)` | Simple baseline; stable gradients early in training |
| **Lovász–BCE** (default) | `0.9 × WeightedBCE + 0.1 × Lovász` | Best for F1/IoU; Lovász directly optimizes a differentiable surrogate of the Jaccard index |
| **Focal–Lovász** | `0.9 × Focal(α=0.75, γ=2) + 0.1 × Lovász` | When easy background pixels dominate; focal loss down-weights confident predictions |


Why `pos_weight = 25`: With ~2% positive pixels, a missed landslide pixel should cost ~25× more than a missed background pixel to balance gradient contributions.

In [ ]:
# BCE, BCE + Lovasz, Focal + Lovasz

def lovasz_grad(gt_sorted):
    p = gt_sorted.numel()
    gts = gt_sorted.sum()

    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted.float()).cumsum(0)

    jaccard = 1.0 - intersection / union.clamp_min(1e-6)

    if p > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]

    return jaccard

def lovasz_hinge_flat(logits, labels):
    if labels.numel() == 0:
        return logits.sum() * 0.0

    labels = labels.float()
    signs = 2.0 * labels - 1.0
    errors = 1.0 - logits * signs

    errors_sorted, perm = torch.sort(errors, descending=True)
    labels_sorted = labels[perm]

    grad = lovasz_grad(labels_sorted)
    return torch.dot(F.relu(errors_sorted), grad)

def batch_lovasz_hinge(logits, targets):
    """
    logits:  [B,H,W]
    targets: [B,H,W]
    """
    losses = [
        lovasz_hinge_flat(logits[i].reshape(-1), targets[i].reshape(-1))
        for i in range(logits.shape[0])
    ]
    return torch.stack(losses).mean()

class WeightedBCELoss(nn.Module):
    """
    Plain weighted BCE baseline.
    """
    def __init__(self, pos_weight=25.0):
        super().__init__()
        self.register_buffer(
            "pos_weight",
            torch.tensor([pos_weight], dtype=torch.float32),
        )

    def forward(self, pred, target):
        logits = pred.squeeze(1)
        target = target.float()

        return F.binary_cross_entropy_with_logits(
            logits,
            target,
            pos_weight=self.pos_weight.to(logits.device),
        )

class LovaszBCELoss(nn.Module):
    """
    Weighted BCE + Lovasz hinge.
    Current strong baseline.
    """
    def __init__(self, bce_weight=0.90, pos_weight=25.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.register_buffer(
            "pos_weight",
            torch.tensor([pos_weight], dtype=torch.float32),
        )

    def forward(self, pred, target):
        logits = pred.squeeze(1)
        target = target.float()

        bce = F.binary_cross_entropy_with_logits(
            logits,
            target,
            pos_weight=self.pos_weight.to(logits.device),
        )

        lovasz = batch_lovasz_hinge(logits, target)

        return self.bce_weight * bce + (1.0 - self.bce_weight) * lovasz

class FocalLovaszLoss(nn.Module):
    """
    Focal loss + Lovasz hinge.
    Useful when easy background pixels dominate training.
    """
    def __init__(
        self,
        focal_weight=0.90,
        alpha=0.75,
        gamma=2.0,
    ):
        super().__init__()
        self.focal_weight = focal_weight
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        logits = pred.squeeze(1)
        target = target.float()

        bce = F.binary_cross_entropy_with_logits(
            logits,
            target,
            reduction="none",
        )

        prob = torch.sigmoid(logits)
        p_t = prob * target + (1.0 - prob) * (1.0 - target)
        alpha_t = self.alpha * target + (1.0 - self.alpha) * (1.0 - target)

        focal = alpha_t * (1.0 - p_t).pow(self.gamma) * bce
        focal = focal.mean()

        lovasz = batch_lovasz_hinge(logits, target)

        return self.focal_weight * focal + (1.0 - self.focal_weight) * lovasz

def build_loss(
    loss_name="bce_lovasz",
    pos_weight=25.0,
    bce_weight=0.90,
    focal_weight=0.90,
    focal_alpha=0.75,
    focal_gamma=2.0,
):
    """
    Available loss_name values:
    - "bce"
    - "bce_lovasz"
    - "focal_lovasz"
    """
    loss_name = loss_name.lower()

    if loss_name == "bce":
        return WeightedBCELoss(pos_weight=pos_weight)

    if loss_name == "bce_lovasz":
        return LovaszBCELoss(
            bce_weight=bce_weight,
            pos_weight=pos_weight,
        )

    if loss_name == "focal_lovasz":
        return FocalLovaszLoss(
            focal_weight=focal_weight,
            alpha=focal_alpha,
            gamma=focal_gamma,
        )

    raise ValueError(
        f"Unknown loss_name='{loss_name}'. "
        "Choose from: 'bce', 'bce_lovasz', 'focal_lovasz'."
    )

## Shared Helpers

**LoRA helper** (next cell): Wraps selected Clay encoder layers with low-rank adapters. Key settings:
- `rank = 4–8`: bottleneck dimension of the low-rank decomposition (higher = more capacity but more parameters)
- Targets: attention Q/K/V projections (`to_qkv`, `to_out`) and MLP layers (`net.1`, `net.3`)
- Optional `layers` filter to restrict LoRA to specific transformer blocks (e.g.last 4 only)

**Utility blocks:** `GN(ch)` provides GroupNorm (batch-size-invariant, important since batch_size=8 is small). `ConvGNReLU` is the reusable `Conv → GroupNorm → ReLU → Dropout` block.

In [ ]:
# ============================================================
# @title LoRA helper
# ============================================================
import re

def apply_lora_to_clay_encoder(encoder, rank=4, alpha=8, layers=None, target_modules=None):
    """
    Wrap selected Linear layers in the Clay encoder with LoRA adapters.

    Args:
        encoder:        Clay encoder module.
        rank:           LoRA rank.
        layers:         Optional list of transformer block indices to target
                        (e.g. [20,21,22,23]).  None = all blocks.
        target_modules: Optional tuple of module-name suffixes to target
                        (e.g. ('to_qkv', 'to_out')).  None = attention + MLP.
    """
    # Save patch_size before PEFT wrapping (it can get lost).
    if hasattr(encoder, 'patch_size'):
        _patch_size = encoder.patch_size

    # Default targets: attention + MLP.
    if target_modules is None:
        target_modules = ('to_qkv', 'to_out', 'net.1', 'net.3')

    # Regex to extract block index from names like
    # "transformer.layers.20.0.to_qkv" or "transformer.layers.20.1.net.1"
    block_re = re.compile(r'transformer\.layers\.(\d+)\.')

    matched = []
    for name, module in encoder.named_modules():
        if not isinstance(module, nn.Linear):
            continue

        # Check suffix match.
        if not any(name.endswith(t) for t in target_modules):
            continue

        # Check block-index filter.
        if layers is not None:
            m = block_re.search(name)
            if m is None:
                continue                    # not inside a transformer block
            if int(m.group(1)) not in layers:
                continue

        matched.append(name)

    matched = sorted(set(matched))

    if not matched:
        raise RuntimeError(
            f'No target layers matched (layers={layers}, target_modules={target_modules}). '
            f'Available Linear layers: '
            f'{[n for n, m in encoder.named_modules() if isinstance(m, nn.Linear)]}'
        )

    print(f'\nLoRA targeting {len(matched)} modules (rank={rank}, alpha={alpha}):')
    for n in matched:
        print('  ', n)

    config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        target_modules=matched,
        lora_dropout=0.1,
        bias='none',
    )

    lora_encoder = get_peft_model(encoder, config)
    lora_encoder.print_trainable_parameters()

    # Restore patch_size if PEFT dropped it.
    if hasattr(encoder, 'patch_size') and not hasattr(lora_encoder, 'patch_size'):
        lora_encoder.patch_size = _patch_size

    return lora_encoder

In [ ]:
# @title Useful Functions

# Utility: GroupNorm helper
def GN(ch):
    return nn.GroupNorm(num_groups=min(8, ch), num_channels=ch)

# Utility: Conv -> GN -> ReLU -> Dropout
class ConvGNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)

# Part 2: Architecture 1 (Clay Backbone)

Clay ViT serves as the primary encoder. A separate lightweight CNN processes terrain (DEM + slope). Features are fused at coarse decoder levels via residual connections with a learnable mixing coefficient.


## Dataloaders

Handle invalid values BEFORE normalization: Find NaN / +Inf / -Inf -> replace value with channel's train mean -> normalize with train std BC/ zero fill distorts value distirbution before normalization (which will ~0 out the values anyway)

In [ ]:
# ============================================================
# Config-driven Dataset
# Reads s2_bands / terrain_bands / input_size from CFG.
# Normalization policy
# Invalid-value policy:
# - Do NOT zero-fill NaN/Inf before normalization
# - Replace invalid values with the TRAIN mean of that band -> normalize with TRAIN std
# ============================================================

# TRAIN-only stats lookup table (channel index e.g. [0] -> {mean,std})
def get_train_band_stats():
    s2s  = TRAIN_NORM_STATS['s2']
    ters = TRAIN_NORM_STATS['terrain']
    return {
        **{i: s2s[L4S_BAND_ORDER[i]] for i in range(12)},
        12: ters['slope'],
        13: ters['dem'],
    }

class Landslide4SenseDataset(Dataset):

    def __init__(self, pairs, split_name, cfg, augment=False):
        """
        Args:
            pairs: list of (img_path, mask_path)
            split_name: kept for compatibility / readability
            cfg: config dict containing selected band indices = flexible for experiments
            augment: whether to apply training-time augmentation
        """
        self.pairs      = pairs # Data file paths (img,msk)
        self.augment    = augment # Augment binary
        self.s2_idx     = cfg['s2_bands'] # Which bands to use
        self.ter_idx    = cfg['terrain_bands']
        self.input_size = cfg['input_size']

        # Loads one fixed set of stats from train
        band_stats = get_train_band_stats()

        # Means/stds for selected S2 bands, shaped as (C,1,1) so they
        # broadcast correctly over chip tensors of shape (C,H,W)
        self.s2_mean = np.array(
            [band_stats[i]['mean'] for i in self.s2_idx], dtype=np.float32
        )[:, None, None]
        self.s2_std = np.array(
            [band_stats[i]['std'] for i in self.s2_idx], dtype=np.float32
        )[:, None, None]

        # Means/stds for selected terrain bands, also shaped (C,1,1)
        self.ter_mean = np.array(
            [band_stats[i]['mean'] for i in self.ter_idx], dtype=np.float32
        )[:, None, None]
        self.ter_std = np.array(
            [band_stats[i]['std'] for i in self.ter_idx], dtype=np.float32
        )[:, None, None]

    def __len__(self):
        return len(self.pairs) # Number of samples = number of file pairs

    def __getitem__(self, idx):
        # Load one img-msk pair from HDF5 based on index of item/pair
        img_p, msk_p = self.pairs[idx]

        with h5py.File(img_p, 'r') as f:
            img = f['img'][:].astype(np.float32)     # expected shape: (128,128,14)

        with h5py.File(msk_p, 'r') as f:
            key = list(f.keys())[0]
            msk = f[key][:].astype(np.int64)         # expected shape: (128,128)

        # Convert image to channel-first format for augmentation / model input
        # from (H,W,C) -> (C,H,W)
        chip = img.transpose(2, 0, 1)                # (14,128,128)

        # Apply augmentation before normalization
        if self.augment:
            chip, msk = augment_geometric(chip, msk)

        # Select requested bands from config
        s2_chip_raw  = chip[self.s2_idx].copy()
        ter_chip_raw = chip[self.ter_idx].copy()

        # Handle invalid values BEFORE normalization
        s2_invalid = ~np.isfinite(s2_chip_raw)
        if np.any(s2_invalid):
            s2_chip_raw[s2_invalid] = np.broadcast_to(self.s2_mean, s2_chip_raw.shape)[s2_invalid]

        ter_invalid = ~np.isfinite(ter_chip_raw)
        if np.any(ter_invalid):
            ter_chip_raw[ter_invalid] = np.broadcast_to(self.ter_mean, ter_chip_raw.shape)[ter_invalid]

        # Normalize using TRAIN-only stats
        s2_chip  = (s2_chip_raw  - self.s2_mean)  / self.s2_std
        ter_chip = (ter_chip_raw - self.ter_mean) / self.ter_std

        # Return tensors:
        # - S2 input tensor
        # - terrain input tensor
        # - segmentation mask
        return (
            torch.from_numpy(s2_chip).float(), # (C, H, W)
            torch.from_numpy(ter_chip).float(),
            torch.from_numpy(msk).long(),
        )

In [ ]:
# PyTorch DataLoaders for training/val/test (turn dataset into batches)
def build_dataloaders(splits, cfg, num_workers=4):
    train_ds = Landslide4SenseDataset(splits['train'],      'train',      cfg, augment=True)
    val_ds   = Landslide4SenseDataset(splits['validation'], 'validation', cfg, augment=False)
    test_ds  = Landslide4SenseDataset(splits['test'],       'test',       cfg, augment=False)

    # Shared setting across all loaders
    dl_common = dict(num_workers=num_workers, pin_memory=True) # pin memory: transfer from CPU to GPU
    if num_workers > 0:
        dl_common.update(dict(persistent_workers=True, prefetch_factor=4)) # Workers stay alive + loads 4 batches ahead

    # Ensure all batches are same size (e.g small last batch) to stablize training
    # randomize order of each epoch
    train_dl = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                          drop_last=True, **dl_common)
    # No shuffling for val/test because no randomness
    val_dl   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                          **dl_common)
    test_dl  = DataLoader(test_ds,  batch_size=cfg['batch_size'], shuffle=False,
                          **dl_common)

    # Dataset size, batches, input size, number of channels, worker setup test
    print(f'Dataset sizes  → Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    print(f'Batch size: {cfg["batch_size"]} | Batches → Train: {len(train_dl)} | Val: {len(val_dl)} | Test: {len(test_dl)}')
    print(f'Input size: {cfg["input_size"]}×{cfg["input_size"]} | S2 ch: {len(cfg["s2_bands"])} | Terrain ch: {len(cfg["terrain_bands"])}')
    print(f'Loader workers: {num_workers} | Persistent workers: {num_workers > 0}')
    return train_dl, val_dl, test_dl

## Model Architecture

This is **version 2** of the Clay backbone architecture. The main changes from version 1 are how terrain is integrated and how the decoder reconstructs spatial resolution.

| Component | Arch 1 v1 (previous) | Arch 1 v2 (this notebook) |
|-----------|---------------------|---------------------------|
| Terrain integration | Early concatenation (terrain appended to Clay input or features once) | **Multi-scale CNN pyramid with residual fusion at 2 decoder levels** |
| Decoder | Pixel-shuffle (single-step upsample from bottleneck) | **3-stage bilinear upsample (16→32→64→128)** |
| Fusion mechanism | Concatenation (terrain treated same as spectral) | **Residual with learnable α (init=0) == terrain as correction term** |
| Loss weighting | Untuned BCE + Lovász | **0.9 × BCE + 0.1 × Lovász (tuned ratio)** |


In [ ]:
# @title Residual Terrain Fusion Block

class ResidualTerrainFusionBlock(nn.Module):
    """
    Lightweight terrain residual fusion
    Terrain acts as a correction term, not a dominant signal

    x_dec: decoder feature
    x_terrain: terrain skip/context feature
    """
    def __init__(self, dec_ch, terrain_ch, out_ch=None, dropout=0.10, init_alpha=0.0):
        super().__init__()

        out_ch = out_ch or dec_ch

        self.dec_proj = nn.Sequential(
            nn.Conv2d(dec_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
        )

        self.terrain_proj = nn.Sequential(
            nn.Conv2d(terrain_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
        )

        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

        self.out = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_dec, x_terrain):
        x_dec = self.dec_proj(x_dec)
        terrain = self.terrain_proj(x_terrain)

        return self.out(x_dec + self.alpha * terrain)

In [ ]:
# @title Terrain CNN Pyramid with Skip Features

# Input: DEM + slope = 2 channels
# Produces terrain skip features at multiple scales.
class TerrainPyramid(nn.Module):
    def __init__(self, in_ch=2, base_ch=96, dropout=0.10):
        super().__init__()

        self.t1 = ConvGNReLU(in_ch,    base_ch, 3, 1, 1, dropout=dropout)  # 128x128
        self.t2 = ConvGNReLU(base_ch,  base_ch, 3, 2, 1, dropout=dropout)  # 64x64
        self.t3 = ConvGNReLU(base_ch,  base_ch, 3, 2, 1, dropout=dropout)  # 32x32
        self.t4 = ConvGNReLU(base_ch,  base_ch, 3, 2, 1, dropout=dropout)  # 16x16

    def forward(self, x):
        t1 = self.t1(x) # [B, C, 128, 128]
        t2 = self.t2(t1) # [B, C, 64, 64]
        t3 = self.t3(t2) # [B, C, 32, 32]
        t4 = self.t4(t3) # [B, C, 16, 16]
        return t1, t2, t3, t4

In [ ]:
# @title Decoder Blocks

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=True),
            GN(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, size):
        x = F.interpolate(x, size=size, mode='bilinear', align_corners=False)
        return self.block(x)

In [ ]:
# @title Define Clay Model
class ClayMultiScaleModel(nn.Module):
    """
    Architecture 1, simplified.

    Design choices:
    - Clay remains the backbone, but the decoder does NOT use Clay encoder skip connections
    - Only the final Clay token map is projected into a CNN bottleneck
    - Terrain branch is a lightweight CNN pyramid with skip features
    - Terrain fusion is residual and only used at coarse decoder levels to reduce high-res terrain noise
    """

    def __init__(
        self,
        clay_encoder,
        embed_dim=1024,
        decoder_dim=96,
        terrain_in_ch=2,
        terrain_fusion_levels=("bottleneck", "d3"),
        terrain_dropout=0.10,
        decoder_dropout=0.10,
        fusion_dropout=0.10,
        terrain_residual_init=0.0,
    ):
        super().__init__()

        self.encoder = clay_encoder
        enc = self.encoder.get_base_model() if hasattr(self.encoder, "get_base_model") else self.encoder
        self.patch_size = enc.patch_size
        self.terrain_fusion_levels = set(terrain_fusion_levels or [])

        self.clay_proj = nn.Sequential(
            nn.Conv2d(embed_dim, decoder_dim, 1, bias=True),
            GN(decoder_dim),
            nn.ReLU(inplace=True),
        )

        self.terrain = TerrainPyramid(
            in_ch=terrain_in_ch,
            base_ch=decoder_dim,
            dropout=terrain_dropout,
        )

        self.fuse_bottleneck = (
            ResidualTerrainFusionBlock(
                dec_ch=decoder_dim,
                terrain_ch=decoder_dim,
                out_ch=decoder_dim,
                dropout=fusion_dropout,
                init_alpha=terrain_residual_init,
            )
            if "bottleneck" in self.terrain_fusion_levels
            else nn.Identity()
        )

        self.up3 = UpBlock(
            in_ch=decoder_dim,
            out_ch=decoder_dim,
            dropout=decoder_dropout,
        )

        self.fuse_d3 = (
            ResidualTerrainFusionBlock(
                dec_ch=decoder_dim,
                terrain_ch=decoder_dim,
                out_ch=decoder_dim,
                dropout=fusion_dropout,
                init_alpha=terrain_residual_init,
            )
            if "d3" in self.terrain_fusion_levels
            else nn.Identity()
        )

        self.up2 = UpBlock(
            in_ch=decoder_dim,
            out_ch=decoder_dim,
            dropout=decoder_dropout,
        )

        self.up1 = UpBlock(
            in_ch=decoder_dim,
            out_ch=decoder_dim,
            dropout=decoder_dropout,
        )

        self.head = nn.Sequential(
            nn.Conv2d(decoder_dim, decoder_dim, 3, padding=1, bias=True),
            GN(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Dropout2d(decoder_dropout) if decoder_dropout > 0 else nn.Identity(),
            nn.Conv2d(decoder_dim, 1, 1, bias=True),
        )

        self._init_new_weights()

        nn.init.normal_(self.head[-1].weight, mean=0.0, std=1e-3)
        nn.init.constant_(self.head[-1].bias, 0.0)

    def _get_encoder_core(self):
        return self.encoder.get_base_model() if hasattr(self.encoder, "get_base_model") else self.encoder

    def _init_new_weights(self):
        enc = self._get_encoder_core()
        encoder_modules = set(enc.modules())

        for m in self.modules():
            if m in encoder_modules:
                continue

            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def _tokens_to_map(self, tokens, H_p, W_p):
        expected = H_p * W_p

        if tokens.shape[1] == expected + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected:
            raise ValueError(
                f"Unexpected token count: got {tokens.shape[1]}, "
                f"expected {expected} or {expected + 1}"
            )

        return rearrange(tokens, "B (H W) D -> B D H W", H=H_p, W=W_p)

    def forward_encoder(self, datacube):
        cube = datacube["pixels"]
        B, _, H, W = cube.shape

        if H % self.patch_size != 0 or W % self.patch_size != 0:
            raise ValueError(
                f"Input size ({H}, {W}) must be divisible by patch_size={self.patch_size}"
            )

        H_p = H // self.patch_size
        W_p = W // self.patch_size
        enc = self._get_encoder_core()

        patches, _ = enc.to_patch_embed(cube, datacube["waves"])
        patches = enc.add_encodings(
            patches,
            datacube["time"],
            datacube["latlon"],
            datacube["gsd"],
        )

        cls = enc.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, patches], dim=1)

        for attn, ff in enc.transformer.layers:
            x = attn(x) + x
            x = ff(x) + x

        return self._tokens_to_map(x, H_p, W_p)

    def _maybe_fuse(self, fusion_layer, x, terrain_skip):
        if isinstance(fusion_layer, nn.Identity):
            return x

        terrain_skip = F.interpolate(
            terrain_skip,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        return fusion_layer(x, terrain_skip)

    def forward(self, datacube, terrain_x):
        s2_x = datacube["pixels"]
        _, _, H, W = s2_x.shape

        clay_final = self.forward_encoder(datacube)
        x = self.clay_proj(clay_final)

        t1, t2, t3, t4 = self.terrain(terrain_x)

        x = self._maybe_fuse(self.fuse_bottleneck, x, t4)

        x = self.up3(x, t3.shape[-2:])
        x = self._maybe_fuse(self.fuse_d3, x, t3)

        x = self.up2(x, t2.shape[-2:])
        x = self.up1(x, t1.shape[-2:])

        x = F.interpolate(
            x,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

        return self.head(x)

In [ ]:
# ============================================================
# @title Model factory: builds ClayMultiScaleModel
# ============================================================
def set_lora_trainable(model, trainable=True):
    """Enable/disable LoRA adapter gradients without touching frozen base Clay weights."""
    for name, param in model.named_parameters():
        if 'lora_' in name:
            param.requires_grad = trainable


def build_model(cfg, ckpt_path):
    # Wavelengths for selected S2 bands only
    waves = torch.tensor(
        [L4S_WAVELENGTH[L4S_BAND_ORDER[i]] for i in cfg['s2_bands']],
        dtype=torch.float32
    )

    # Build Clay segmentor only to load pretrained encoder.
    seg = Segmentor(num_classes=1, ckpt_path=ckpt_path)

    if cfg['model'] == 'lora':
        print('Applying LoRA to Clay encoder...')
        seg.encoder = apply_lora_to_clay_encoder(
            seg.encoder,
            rank=cfg['lora_rank'],
            alpha=cfg['lora_alpha']
        )

        # Always freeze the pretrained base encoder weights.
        for name, param in seg.encoder.named_parameters():
            param.requires_grad = ('lora_' in name)

        # Optional stage 1: keep LoRA frozen until decoder stabilizes.
        if cfg.get('two_stage', False) and cfg.get('freeze_encoder_epochs', 0) > 0:
            for name, param in seg.encoder.named_parameters():
                if 'lora_' in name:
                    param.requires_grad = False
            print(f'LoRA adapters frozen for stage 1 ({cfg["freeze_encoder_epochs"]} epochs).')
    else:
        print('Using frozen Clay encoder...')
        for param in seg.encoder.parameters():
            param.requires_grad = False

    model = ClayMultiScaleModel(
        clay_encoder=seg.encoder,
        embed_dim=cfg.get('embed_dim', 1024),
        decoder_dim=cfg.get('decoder_dim', 96),
        terrain_in_ch=len(cfg.get('terrain_bands', [12, 13])),
        terrain_fusion_levels=cfg.get('terrain_fusion_levels', ("bottleneck", "d3")),
        terrain_dropout=cfg.get('terrain_dropout', 0.10),
        decoder_dropout=cfg.get('decoder_dropout', 0.10),
        fusion_dropout=cfg.get('fusion_dropout', 0.10),
    )

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'Total params     : {total:,}')
    print(f'Trainable params : {trainable:,}  ({100 * trainable / total:.1f}%)')
    print(f'Frozen params    : {total - trainable:,}')
    print(f'Wavelengths fed to Clay: {waves.tolist()}')
    print(f'Terrain fusion levels: {cfg.get("terrain_fusion_levels", ("bottleneck", "d3"))}')

    model.register_buffer("clay_waves", waves)
    model = model.to(device)
    return model

## Training Configuration & Execution

The experiment registry below defines all hyperparameters for each run. Switching experiments requires changing only `ACTIVE_EXP`. Key differences between experiments:

- `arch1_frozen_simple`: Frozen Clay encoder, only decoder trains. Single-stage. Baseline for measuring LoRA's benefit.
- `arch1_lora_twostage_simple`: Two-stage training: Stage 1 (epochs 1–10) freezes LoRA and trains only the decoder + terrain; Stage 2 (epoch 11+) unfreezes LoRA atlower LR. This stabilizes the decoder before allowing encoder adaptation.

All experiments have early stopping and dropout (=0.10).

In [ ]:
# ============================================================
# @title EXPERIMENT REGISTRY
# Band indices into the 14-band chip:
#   0=B1  1=B2  2=B3  3=B4  4=B5   5=B6   6=B7  7=B8
#   8=B9  9=B10 10=B11 11=B12 12=Slope 13=DEM
# ============================================================
EXPERIMENTS = {

    'arch1_frozen_simple': {
        'description': 'Architecture 1 simplified: frozen Clay + decoder, coarse terrain fusion',
        'model': 'frozen',
        'input_size': 128,
        's2_bands': list(range(12)),
        'terrain_bands': [12, 13],
        'loss': 'bce_lovasz',
        'batch_size': 8,
        'epochs': 50,
        'patience': 10,

        'decoder_dim': 96,
        'terrain_fusion_levels': ('bottleneck', 'd3'),
        'terrain_dropout': 0.10,
        'decoder_dropout': 0.10,
        'fusion_dropout': 0.10,

        'lr_decoder': 3e-4,
        'lr_lora': None,
        'weight_decay': 1e-3,

        # no two-stage
        'use_two_stage': False,
    },

    'arch1_lora_twostage_simple': {
        'description': 'Architecture 1: LoRA + two-stage + coarse terrain fusion',
        'model': 'lora',
        'input_size': 128,
        's2_bands': list(range(12)),
        'terrain_bands': [12, 13],
        'loss': 'bce_lovasz',

        'batch_size': 8,
        'epochs': 50,
        'patience': 10,

        'lora_rank': 8,
        'lora_alpha': 8,
        'decoder_dim': 96,

        'terrain_fusion_levels': ('bottleneck', 'd3'),
        'terrain_dropout': 0.10,
        'decoder_dropout': 0.10,
        'fusion_dropout': 0.10,

        # Stage 1
        'lr_decoder': 3e-4,
        'lr_lora': None,

        # Stage 2 (FIXED KEYS)
        'use_two_stage': True,
        'stage2_start_epoch': 11,
        'stage2_train_lora': True,
        'stage2_lr_decoder': 1e-4,
        'stage2_lr_lora': 1e-5,

        'weight_decay': 1e-3,
        'min_lr': 1e-6,
    },
}

# !!! ONLY CHANGE THIS LINE TO SWITCH EXPERIMENTS
ACTIVE_EXP = 'arch1_lora_twostage_simple'

CFG = EXPERIMENTS[ACTIVE_EXP]
print(f'Active experiment : {ACTIVE_EXP}')
print(f'Description       : {CFG["description"]}')
print(f'Model             : {CFG["model"]}')
print(f'Input size        : {CFG["input_size"]}x{CFG["input_size"]}')
print(f'S2 bands          : {CFG["s2_bands"]}  ({len(CFG["s2_bands"])} ch)')
print(f'Terrain bands     : {CFG["terrain_bands"]}  ({len(CFG["terrain_bands"])} ch)')
print(f'Terrain fusion    : {CFG.get("terrain_fusion_levels", ())}')
print(f'Loss              : {CFG["loss"]}')
print(f'Two-stage         : {CFG.get("two_stage", False)} | freeze epochs: {CFG.get("freeze_encoder_epochs", 0)}')
print(f'Batch size        : {CFG["batch_size"]}')
print(f'Epochs / patience : {CFG["epochs"]} / {CFG["patience"]}')

* Even with Lovasz loss, the evaluation metric is based on F1 not just validation loss, thus, early stopping should be consistent.

In [ ]:
# @title Build Datacube
# Set the specifica settings for training
def build_clay_datacube(s2_x, waves, clay_time, clay_latlon, clay_gsd):
    B = s2_x.shape[0]

    return {
        "pixels": s2_x,                              # (B, C, H, W)
        "time": clay_time.expand(B, -1),             # (B, 4)
        "latlon": clay_latlon.expand(B, -1),         # (B, 4)
        "gsd": clay_gsd,
        "waves": waves,                              # (C,)
    }

In [ ]:
# @title Define Epoch Runner
def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    phase="train",
    scaler=None,
    threshold=0.5,
):
    is_train = phase == "train"
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_tp, total_fp, total_fn = 0, 0, 0
    total_prob_sum, total_prob_count = 0.0, 0

    sz = CFG.get("input_size", 128)
    use_amp = device.type == "cuda"

    grad_ctx = torch.enable_grad() if is_train else torch.no_grad()

    with grad_ctx:
        for s2_x, terrain_x, y in loader:
            s2_x = s2_x.to(device, non_blocking=True)
            terrain_x = terrain_x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()

            waves = getattr(model, "clay_waves", CLAY_WAVES.to(device))

            datacube = build_clay_datacube(
                s2_x,
                waves,
                CLAY_TIME.to(device),
                CLAY_LATLON.to(device),
                CLAY_GSD.to(device),
            )

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(datacube, terrain_x)

                if logits.shape[-2:] != y.shape[-2:]:
                    logits = F.interpolate(
                        logits,
                        size=y.shape[-2:],
                        mode="bilinear",
                        align_corners=False,
                    )

                logits = torch.clamp(logits, -20, 20)
                loss = criterion(logits, y)

            if is_train:
                if scaler is not None and use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            total_loss += loss.item()

            with torch.no_grad():
                logits_det = logits.detach()
                if logits_det.ndim == 4:
                    logits_det = logits_det.squeeze(1)

                probs = torch.sigmoid(logits_det)
                preds = probs >= threshold
                y_bool = y > 0.5

                total_tp += (preds & y_bool).sum().item()
                total_fp += (preds & ~y_bool).sum().item()
                total_fn += (~preds & y_bool).sum().item()

                total_prob_sum += probs.sum().item()
                total_prob_count += probs.numel()

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-8)
    iou = total_tp / (total_tp + total_fp + total_fn + 1e-8)

    return {
        "loss": total_loss / max(len(loader), 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "mean_pos_prob": total_prob_sum / max(total_prob_count, 1),
    }

In [ ]:
# @title Optimizer + Two-Stage Trainability
def is_lora_param(name):
    name = name.lower()
    return "lora_" in name or ".lora_" in name

def set_lora_trainable(model, trainable=True):
    for name, param in model.named_parameters():
        if is_lora_param(name):
            param.requires_grad = trainable

def freeze_encoder_except_lora(model):
    """
    Freeze Clay/base encoder params.
    Keep decoder, terrain branch, fusion layers, and head trainable.
    LoRA trainability is controlled separately.
    """
    enc = model._get_encoder_core() if hasattr(model, "_get_encoder_core") else model.encoder
    encoder_param_ids = {id(p) for p in enc.parameters()}

    for p in model.parameters():
        if id(p) in encoder_param_ids:
            p.requires_grad = False
        else:
            p.requires_grad = True

def configure_stage_trainability(
    model,
    stage=1,
    train_lora_stage2=True,
):
    """
    Stage 1:
    - Freeze Clay encoder completely, including LoRA.
    - Train decoder/fusion/terrain/head.

    Stage 2:
    - Keep base Clay frozen.
    - Optionally unfreeze LoRA adapters.
    - Continue training decoder/fusion/terrain/head.
    """
    freeze_encoder_except_lora(model)

    if stage == 1:
        set_lora_trainable(model, trainable=False)

    elif stage == 2:
        set_lora_trainable(model, trainable=train_lora_stage2)

    else:
        raise ValueError(f"stage must be 1 or 2, got {stage}")

def make_optimizer_scheduler(
    model,
    lr_decoder=3e-4,
    lr_lora=None,
    weight_decay=1e-3,
    epochs=50,
    min_lr=1e-6,
):
    """
    Build optimizer from currently trainable params.

    Important:
    Recreate this optimizer after changing requires_grad,
    especially when entering Stage 2 and unfreezing LoRA.
    """
    lr_lora = lr_decoder if lr_lora is None else lr_lora

    lora_params = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if is_lora_param(name):
            lora_params.append(param)
        else:
            other_params.append(param)

    param_groups = []

    if other_params:
        param_groups.append({
            "params": other_params,
            "lr": lr_decoder,
            "name": "decoder_fusion_terrain_head",
        })

    if lora_params:
        param_groups.append({
            "params": lora_params,
            "lr": lr_lora,
            "name": "lora",
        })

    if not param_groups:
        raise RuntimeError("No trainable parameters found. Check requires_grad settings.")

    optimizer = torch.optim.AdamW(
        param_groups,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, epochs),
        eta_min=min_lr,
    )

    return optimizer, scheduler

def maybe_switch_to_stage2(
    model,
    epoch,
    optimizer,
    scheduler,
    cfg,
):
    """
    Switch to Stage 2 exactly once.

    Required CFG keys:
    - use_two_stage
    - stage2_start_epoch
    - stage2_train_lora
    - stage2_lr_decoder
    - stage2_lr_lora
    - weight_decay
    - epochs
    """
    if not cfg.get("use_two_stage", True):
        return optimizer, scheduler, 1

    stage2_start = int(cfg.get("stage2_start_epoch", 12))

    if epoch != stage2_start:
        current_stage = 1 if epoch < stage2_start else 2
        return optimizer, scheduler, current_stage

    configure_stage_trainability(
        model,
        stage=2,
        train_lora_stage2=cfg.get("stage2_train_lora", True),
    )
    remaining_epochs = max(1, int(cfg["epochs"]) - epoch + 1)

    optimizer, scheduler = make_optimizer_scheduler(
        model,
        lr_decoder=cfg.get("stage2_lr_decoder", 1e-4),
        lr_lora=cfg.get("stage2_lr_lora", 1e-5),
        weight_decay=cfg.get("weight_decay", 1e-3),
        epochs=remaining_epochs,
        min_lr=cfg.get("min_lr", 1e-6),
    )

    return optimizer, scheduler, 2

def count_trainable_params(model):
    total = 0
    lora = 0
    other = 0

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        n = param.numel()
        total += n
        if is_lora_param(name):
            lora += n
        else:
            other += n
    return {
        "total_trainable": total,
        "lora_trainable": lora,
        "other_trainable": other,
    }


In [ ]:
# @title Define full training loop

def train_model(
    model,
    name,
    train_dl,
    val_dl,
    loss_fn="bce_lovasz",
    epochs=50,
    patience=10,
    save_dir=DRIVE_DIR,
    threshold=0.5,
):
    criterion = build_loss(
        loss_name=loss_fn,
        pos_weight=CFG.get("pos_weight", 25.0),
        bce_weight=CFG.get("bce_weight", 0.90),
        focal_weight=CFG.get("focal_weight", 0.90),
        focal_alpha=CFG.get("focal_alpha", 0.75),
        focal_gamma=CFG.get("focal_gamma", 2.0),
    ).to(device)

    use_two_stage = CFG.get("use_two_stage", True)
    stage2_start = int(CFG.get("stage2_start_epoch", 12))

    if use_two_stage:
        configure_stage_trainability(
            model,
            stage=1,
            train_lora_stage2=CFG.get("stage2_train_lora", True),
        )
        opt_epochs = max(1, stage2_start - 1)
    else:
        configure_stage_trainability(
            model,
            stage=2,
            train_lora_stage2=CFG.get("stage2_train_lora", True),
        )
        opt_epochs = epochs

    optimizer, scheduler = make_optimizer_scheduler(
        model,
        lr_decoder=CFG.get("lr_decoder", 3e-4),
        lr_lora=CFG.get("lr_lora", None),
        weight_decay=CFG.get("weight_decay", 1e-3),
        epochs=opt_epochs,
        min_lr=CFG.get("min_lr", 1e-6),
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    history = {
        "train_loss": [], "val_loss": [],
        "train_precision": [], "val_precision": [],
        "train_recall": [], "val_recall": [],
        "train_f1": [], "val_f1": [],
        "train_iou": [], "val_iou": [],
        "train_mean_pos_prob": [], "val_mean_pos_prob": [],
        "lr_decoder": [], "lr_lora": [],
        "stage": [],
    }

    best_val_f1 = -1.0
    best_epoch = 0
    patience_count = 0
    save_path = f"{save_dir}/{name}_best.pt"

    print(f"Using loss: {loss_fn}")
    print(criterion)
    print(f"\n=== Training {name} | loss={loss_fn} | select_by=val_f1 ===")
    print(
        f'{"Epoch":>5} {"Stage":>5} {"TrLoss":>8} {"VaLoss":>8} '
        f'{"TrF1":>7} {"VaF1":>7} {"VaIoU":>7} {"VaP":>7} {"VaR":>7} '
        f'{"VaProb":>8} {"LRd":>9} {"LRl":>9} {"Time":>6}'
    )
    print("-" * 112)

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        optimizer, scheduler, stage = maybe_switch_to_stage2(
            model=model,
            epoch=epoch,
            optimizer=optimizer,
            scheduler=scheduler,
            cfg={**CFG, "epochs": epochs},
        )

        if use_two_stage and epoch == stage2_start:
            patience_count = 0
            print(f"Switched to Stage 2 at epoch {epoch}; patience reset.")

        train_stats = run_epoch(
            model, train_dl, criterion,
            optimizer=optimizer,
            phase="train",
            scaler=scaler,
            threshold=threshold,
        )

        val_stats = run_epoch(
            model, val_dl, criterion,
            optimizer=None,
            phase="val",
            scaler=None,
            threshold=threshold,
        )

        scheduler.step()

        lr_decoder_now = next(
            (pg["lr"] for pg in optimizer.param_groups if pg.get("name") == "decoder_fusion_terrain_head"),
            None,
        )
        lr_lora_now = next(
            (pg["lr"] for pg in optimizer.param_groups if pg.get("name") == "lora"),
            None,
        )

        for k in ["loss", "precision", "recall", "f1", "iou", "mean_pos_prob"]:
            history[f"train_{k}"].append(train_stats[k])
            history[f"val_{k}"].append(val_stats[k])

        history["lr_decoder"].append(lr_decoder_now)
        history["lr_lora"].append(lr_lora_now)
        history["stage"].append(stage)

        improved = val_stats["f1"] > best_val_f1

        if improved:
            best_val_f1 = val_stats["f1"]
            best_epoch = epoch
            patience_count = 0
            torch.save(model.state_dict(), save_path)
            flag = " *"
        else:
            patience_count += 1
            flag = ""

        print(
            f'{epoch:>5} {stage:>5} '
            f'{train_stats["loss"]:>8.4f} {val_stats["loss"]:>8.4f} '
            f'{train_stats["f1"]:>7.4f} {val_stats["f1"]:>7.4f} '
            f'{val_stats["iou"]:>7.4f} {val_stats["precision"]:>7.4f} {val_stats["recall"]:>7.4f} '
            f'{val_stats["mean_pos_prob"]:>8.4f} '
            f'{(lr_decoder_now or 0):>9.2e} '
            f'{(lr_lora_now or 0):>9.2e} '
            f'{time.time() - t0:>6.0f}s{flag}'
        )

        can_early_stop = (not use_two_stage) or (epoch >= stage2_start)

        if can_early_stop and patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_epoch > 0:
        model.load_state_dict(torch.load(save_path, map_location=device))

    print(f"\nBest val F1: {best_val_f1:.4f} at epoch {best_epoch}  ->  {save_path}")

    history["best_val_f1"] = best_val_f1
    history["best_epoch"] = best_epoch
    history["best_ckpt"] = save_path

    return history

* Probabilities at true POS/NEG = collects model confidence on true landslide pixels vs true background pixels

In [ ]:
# ============================================================
# Collect probabilities + targets from a loader
# Used for validation threshold selection and fixed-threshold test eval
# ============================================================
def collect_probs_and_targets(model, loader):
    model.eval()

    probs_list = []
    targets_list = []
    sz = CFG.get("input_size", 128)

    with torch.no_grad():
        for s2_x, terrain_x, y in loader:
            s2_x = s2_x.to(device, non_blocking=True)
            terrain_x = terrain_x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()

            waves = getattr(model, "clay_waves", CLAY_WAVES.to(device))

            datacube = build_clay_datacube(
                s2_x,
                waves,
                CLAY_TIME.to(device),
                CLAY_LATLON.to(device),
                CLAY_GSD.to(device),
            )
            logits = model(datacube, terrain_x)

            if logits.shape[-2:] != y.shape[-2:]:
                logits = F.interpolate(
                    logits,
                    size=y.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )

            probs = torch.sigmoid(logits).squeeze(1)
            probs_list.append(probs.cpu().numpy().astype(np.float32))
            targets_list.append(y.cpu().numpy().astype(np.uint8))

    probs = np.concatenate(probs_list, axis=0)
    targets = np.concatenate(targets_list, axis=0)

    return probs, targets

In [ ]:
# ============================================================
# Sweep threshold on validation only
# ============================================================
def sweep_threshold_from_probs(probs, targets, thresholds=None):
    if thresholds is None:
        thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

    pos = probs[targets.astype(bool)]
    neg = probs[~targets.astype(bool)]

    print(f'Positive pixels : {len(pos):,}  mean prob {pos.mean():.3f}')
    print(f'Negative pixels : {len(neg):,}  mean prob {neg.mean():.3f}')
    print(f'\n{"Thresh":>8} {"TP":>9} {"FP":>9} {"FN":>9} '
          f'{"IoU":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
    print('-' * 74)

    best_f1 = -1.0
    best_thresh = 0.5
    results = []

    for t in thresholds:
        tp = int((pos >= t).sum()) # tp if positive pixels
        fp = int((neg >= t).sum())
        fn = int((pos < t).sum())

        iou  = tp / (tp + fp + fn + 1e-8)
        prec = tp / (tp + fp + 1e-8)
        rec  = tp / (tp + fn + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)

        row = dict(thresh=t, iou=iou, prec=prec, rec=rec, f1=f1)
        results.append(row)

        print(f'{t:>8.2f} {tp:>9,} {fp:>9,} {fn:>9,} '
              f'{iou:>8.3f} {prec:>8.3f} {rec:>8.3f} {f1:>8.3f}')

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print(f'\nBest validation threshold: {best_thresh} | F1={best_f1:.3f}')
    return {
        'best_f1': float(best_f1),
        'best_thresh': float(best_thresh),
        'sweep': results,
        'pos_mean': float(pos.mean()),
        'neg_mean': float(neg.mean()),
    }

**Design Note:** for evaluation protocol, threshold tuning matter.

Model outputs logits → sigmoid → probabilities. Converting to binary predictions requires a threshold. With 2% class imbalance, the optimal threshold is often far from 0.5

Thus, implement protocol:
1. Collect all predicted probabilities on the **validation** set
2. Sweep thresholds (0.1 to 0.8) and pick the one maximizing val F1
3. Lock that threshold and evaluate test set exactly once

This prevents test-set contamination. Used for both Arch 1 and Arch 2


In [ ]:
# ============================================================
# Evaluate once on test using validation-chosen threshold
# ============================================================
def evaluate_fixed_threshold(model, loader, name, threshold):
    probs, targets = collect_probs_and_targets(model, loader)

    preds = (probs >= threshold).astype(np.uint8)
    targets = targets.astype(np.uint8)

    tp = int(((preds == 1) & (targets == 1)).sum())
    fp = int(((preds == 1) & (targets == 0)).sum())
    fn = int(((preds == 0) & (targets == 1)).sum())

    iou  = tp / (tp + fp + fn + 1e-8)
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)

    print(f'\n=== {name} — fixed-threshold evaluation ===')
    print(f'Threshold : {threshold:.2f}')
    print(f'TP        : {tp:,}')
    print(f'FP        : {fp:,}')
    print(f'FN        : {fn:,}')
    print(f'IoU       : {iou:.4f}')
    print(f'Precision : {prec:.4f}')
    print(f'Recall    : {rec:.4f}')
    print(f'F1        : {f1:.4f}')

    return {
        'threshold': float(threshold),
        'iou': float(iou),
        'prec': float(prec),
        'rec': float(rec),
        'f1': float(f1),
        'tp': tp,
        'fp': fp,
        'fn': fn,
    }

In [ ]:
# @title Build Dataloaders + Model
# Instantiates the model (from blueprint in Model Factory) + moves it to GPU

print(f'\n{"="*60}')
print(f'EXPERIMENT: {ACTIVE_EXP}')
print(CFG["description"])
print(f'{"="*60}\n')

train_dl, val_dl, test_dl = build_dataloaders(SPLITS, CFG)

s2_x, terrain_x, y = next(iter(train_dl))
print("Batch check:")
print("  s2_x     :", s2_x.shape)
print("  terrain_x:", terrain_x.shape)
print("  y        :", y.shape)
print("  pos ratio:", y.float().mean().item())

model = build_model(CFG, CKPT_PATH)
model = model.to(device)

# Optional forward check
model.eval()
with torch.no_grad():
    datacube = build_clay_datacube(
        s2_x.to(device),
        getattr(model, "clay_waves", CLAY_WAVES.to(device)),
        CLAY_TIME.to(device),
        CLAY_LATLON.to(device),
        CLAY_GSD.to(device),
    )
    logits = model(datacube, terrain_x.to(device))

print(
    f"Forward pass: logits {logits.shape} "
    f"range [{logits.min():.3f}, {logits.max():.3f}] "
    f"NaN: {logits.isnan().any().item()}"
)

In [ ]:
# @title Run Training
history = train_model(
    model, ACTIVE_EXP,
    train_dl, val_dl,
    loss_fn  = CFG['loss'],
    epochs   = CFG['epochs'],
    patience = CFG['patience'],
    save_dir = DRIVE_DIR
)

# Training curves (Loss + F1)
epochs = np.arange(1, len(history['train_loss']) + 1)
best_epoch = history.get('best_epoch', None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')

if best_epoch is not None:
    axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Chosen epoch ({best_epoch})')

axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_xticks(epochs)
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs, history['train_f1'], label='Train')
axes[1].plot(epochs, history['val_f1'], label='Val')

if best_epoch is not None:
    axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Chosen epoch ({best_epoch})')

    # mark best point
    if 1 <= best_epoch <= len(history['val_f1']):
        best_f1 = history['val_f1'][best_epoch - 1]
        axes[1].scatter(best_epoch, best_f1, color='red', zorder=5)

axes[1].set_title('F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].set_xticks(epochs)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
save_path = f"{DRIVE_DIR}/{ACTIVE_EXP}_training_curves.png"
plt.savefig(save_path, dpi=200, bbox_inches='tight')

print(f"Saved plot to: {save_path}")

plt.show()

In [ ]:
torch.cuda.empty_cache() # clear GPU cache
import gc; gc.collect()

In [ ]:
# @title Run Evaluation
# 1) choose threshold on validation only
# 2) evaluate once on test with locked threshold

print("\nSelecting threshold on VALIDATION only...")
val_probs, val_targets = collect_probs_and_targets(model, val_dl)
val_results = sweep_threshold_from_probs(val_probs, val_targets)

best_thresh = val_results['best_thresh']

print("\nEvaluating on TEST with validation-chosen threshold...")
test_results = evaluate_fixed_threshold(model, test_dl, ACTIVE_EXP, threshold=best_thresh)

summary = {
    'experiment': ACTIVE_EXP,
    'description': CFG['description'],
    'val_best_f1': val_results['best_f1'],
    'val_best_thresh': val_results['best_thresh'],
    'val_pos_mean': val_results['pos_mean'],
    'val_neg_mean': val_results['neg_mean'],
    'val_sweep': val_results['sweep'],
    'test_metrics': test_results,
    'config': CFG,
}

with open(f'{DRIVE_DIR}/{ACTIVE_EXP}_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved -> {DRIVE_DIR}/{ACTIVE_EXP}_results.json')

##  Multi-seed for Model 5
Running to get standard deviation and mean for multiple runs of the best Arch 1 model.

In [ ]:
# Make sure ACTIVE_EXP = 'arch1_lora_twostage_simple' and CFG points to it

def set_all_seeds(seed):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ms(vals):
    a = np.array(vals)
    return float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0

assert ACTIVE_EXP == 'arch1_lora_twostage_simple', \
    f"Set ACTIVE_EXP='arch1_lora_twostage_simple' before running. Current: {ACTIVE_EXP}"

SEEDS_A1 = [2024]   # ← CHANGE PER RESTART: [42], [123], [2024]
ALL_SEEDS_A1 = [42, 123, 2024]   # full set we expect when done

partial_path_a1 = f'{DRIVE_DIR}/{ACTIVE_EXP}_multiseed_partial.json'
if os.path.exists(partial_path_a1):
    with open(partial_path_a1) as f:
        multi_seed_a1 = json.load(f)['results']
    done = {r['seed'] for r in multi_seed_a1}
    print(f'Resuming: have seeds {sorted(done)}')
else:
    multi_seed_a1, done = [], set()

for seed in SEEDS_A1:
    if seed in done:
        print(f'Skip seed {seed} (done)'); continue
    print(f'\n{"="*60}\nARCH 1 — SEED {seed}\n{"="*60}')
    set_all_seeds(seed)

    seed_exp_name = f'{ACTIVE_EXP}_seed{seed}'

    # Rebuild data + model fresh
    train_dl_s, val_dl_s, test_dl_s = build_dataloaders(SPLITS, CFG)
    model_s = build_model(CFG, CKPT_PATH).to(device)

    _ = train_model(
        model_s, seed_exp_name,
        train_dl_s, val_dl_s,
        loss_fn  = CFG['loss'],
        epochs   = CFG['epochs'],
        patience = CFG['patience'],
        save_dir = DRIVE_DIR
    )

    # Validation threshold sweep
    val_probs_s, val_tgts_s = collect_probs_and_targets(model_s, val_dl_s)
    val_results_s = sweep_threshold_from_probs(val_probs_s, val_tgts_s)
    best_t_s = val_results_s['best_thresh']

    # Test at locked threshold + at t=0.50
    test_locked = evaluate_fixed_threshold(model_s, test_dl_s, seed_exp_name, threshold=best_t_s)
    test_050    = evaluate_fixed_threshold(model_s, test_dl_s, seed_exp_name, threshold=0.50)

    multi_seed_a1.append({
        'seed': seed, 'best_threshold': float(best_t_s),
        'test_precision': float(test_locked['prec']),
        'test_recall':    float(test_locked['rec']),
        'test_f1':        float(test_locked['f1']),
        'test_precision_t050': float(test_050['prec']),
        'test_recall_t050':    float(test_050['rec']),
        'test_f1_t050':        float(test_050['f1']),
    })
    print(f"Seed {seed}: F1@locked(t={best_t_s:.2f})={test_locked['f1']:.4f}  |  F1@0.50={test_050['f1']:.4f}")

    with open(partial_path_a1, 'w') as f:
        json.dump({'results': multi_seed_a1}, f, indent=2)

    del model_s, train_dl_s, val_dl_s, test_dl_s, val_probs_s, val_tgts_s
    torch.cuda.empty_cache()
    import gc; gc.collect()


In [ ]:
# ----------------------------------------------------------
# Aggregate ONLY when all expected seeds are complete
# ----------------------------------------------------------
completed_seeds = {r['seed'] for r in multi_seed_a1}
if set(ALL_SEEDS_A1).issubset(completed_seeds):
    agg_a1 = {k: dict(zip(['mean','std'], ms([r[k] for r in multi_seed_a1])))
              for k in ['test_precision','test_recall','test_f1',
                        'test_precision_t050','test_recall_t050','test_f1_t050','best_threshold']}

    print(f'\n=== ARCH 1 Multi-seed Summary (n={len(multi_seed_a1)}) ===')
    print(f"Locked t (mean={agg_a1['best_threshold']['mean']:.2f} ± {agg_a1['best_threshold']['std']:.2f}):")
    print(f"  P  = {agg_a1['test_precision']['mean']*100:.1f} ± {agg_a1['test_precision']['std']*100:.1f}")
    print(f"  R  = {agg_a1['test_recall']['mean']*100:.1f} ± {agg_a1['test_recall']['std']*100:.1f}")
    print(f"  F1 = {agg_a1['test_f1']['mean']*100:.1f} ± {agg_a1['test_f1']['std']*100:.1f}")
    print(f"At t=0.50:")
    print(f"  P  = {agg_a1['test_precision_t050']['mean']*100:.1f} ± {agg_a1['test_precision_t050']['std']*100:.1f}")
    print(f"  R  = {agg_a1['test_recall_t050']['mean']*100:.1f} ± {agg_a1['test_recall_t050']['std']*100:.1f}")
    print(f"  F1 = {agg_a1['test_f1_t050']['mean']*100:.1f} ± {agg_a1['test_f1_t050']['std']*100:.1f}")

    with open(f'{DRIVE_DIR}/arch1_multiseed_final.json','w') as f:
        json.dump({'seeds': sorted(completed_seeds),
                   'per_seed': multi_seed_a1, 'aggregate': agg_a1}, f, indent=2)
    print('Saved arch1_multiseed_final.json')
else:
    remaining = sorted(set(ALL_SEEDS_A1) - completed_seeds)
    print(f'\n{"="*60}')
    print(f'Completed seeds: {sorted(completed_seeds)}')
    print(f'Remaining seeds: {remaining}')
    print(f'Aggregate skipped — will compute once all seeds finish.')
    print(f'Next: restart runtime, set SEEDS_A1 = [{remaining[0]}], rerun this cell.')
    print(f'{"="*60}')

# Part 3: Baseline (U-Net Comparison)

Standard U-Net adapted from the original Landslide4Sense competition code: same channel widths, same bilinear upsampling, same 2-class softmax output, same cross-entropy loss, same Adam optimizer at LR=1e-3

**Key differences:** U-Net uses the same train-only normalization and geometric augmentation pipeline as Architectures 1 and 2. This makes it a slightly stronger baseline than the raw competition code, but ensures the comparison isolates the effect of the encoder (Clay vs. CNN) rather than preprocessing differences.



In [ ]:
# ============================================================
# Competition U-Net baseline on current data pipeline
# - Uses SPLITS and TRAIN_NORM_STATS
# - Uses THEIR exact U-Net architecture style
# - Uses THEIR 2-class CE training/eval logic
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epsilon = 1e-14

# ------------------------------------------------------------
# 1) Exact competition U-Net architecture
#    Adapted directly from Networks.py
# ------------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [
            diffX // 2, diffX - diffX // 2,
            diffY // 2, diffY - diffY // 2
        ])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class CompetitionUNet(nn.Module):
    def __init__(self, n_classes=2, n_channels=14, bilinear=True):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

# ------------------------------------------------------------
# 2) 14-channel stats lookup using MY train-only stats
# ------------------------------------------------------------
def get_train_band_stats_14():
    s2s = TRAIN_NORM_STATS['s2']
    ters = TRAIN_NORM_STATS['terrain']
    return {
        0:  s2s['coastal'],
        1:  s2s['blue'],
        2:  s2s['green'],
        3:  s2s['red'],
        4:  s2s['rededge1'],
        5:  s2s['rededge2'],
        6:  s2s['rededge3'],
        7:  s2s['nir'],
        8:  s2s['water_vapor'],
        9:  s2s['cirrus'],
        10: s2s['swir16'],
        11: s2s['swir22'],
        12: ters['slope'],
        13: ters['dem'],
    }

# ------------------------------------------------------------
# 3) Dataset: uses MY paired SPLITS and MY normalization
#    Returns one 14-channel tensor exactly for competition U-Net
# ------------------------------------------------------------
class L4SCompetitionDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = pairs
        self.augment = augment

        band_stats = get_train_band_stats_14()
        self.mean = np.array([band_stats[i]['mean'] for i in range(14)], dtype=np.float32)[:, None, None]
        self.std  = np.array([band_stats[i]['std']  for i in range(14)], dtype=np.float32)[:, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, msk_p = self.pairs[idx]

        with h5py.File(img_p, 'r') as f:
            img = f['img'][:].astype(np.float32)   # (H,W,14)

        with h5py.File(msk_p, 'r') as f:
            key = list(f.keys())[0]
            msk = f[key][:].astype(np.int64)       # (H,W)

        chip = img.transpose(2, 0, 1)             # (14,H,W)

        # Keep YOUR current geometric augmentation only
        if self.augment:
            chip, msk = augment_geometric(chip, msk)

        # Fill invalid values with train mean, then normalize
        invalid = ~np.isfinite(chip)
        if np.any(invalid):
            chip[invalid] = np.broadcast_to(self.mean, chip.shape)[invalid]

        chip = (chip - self.mean) / self.std

        # Force labels to {0,1}
        msk = (msk > 0).astype(np.int64)

        return (
            torch.from_numpy(chip).float(),   # (14,H,W)
            torch.from_numpy(msk).long(),     # (H,W)
            os.path.basename(img_p),
        )

def build_competition_dataloaders(splits, batch_size=8, num_workers=4):
    train_ds = L4SCompetitionDataset(splits['train'], augment=True)
    val_ds   = L4SCompetitionDataset(splits['validation'], augment=False)
    test_ds  = L4SCompetitionDataset(splits['test'], augment=False)

    dl_common = dict(num_workers=num_workers, pin_memory=True)
    if num_workers > 0:
        dl_common.update(dict(persistent_workers=True, prefetch_factor=4))

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **dl_common)
    val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, **dl_common)
    test_dl  = DataLoader(test_ds, batch_size=1, shuffle=False, **dl_common)

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    print(f"Train batches: {len(train_dl)} | batch_size={batch_size}")
    return train_dl, val_dl, test_dl

# ------------------------------------------------------------
# 4) Exact competition-style metric accumulation
#    Mirrors tools.py / Train.py logic
# ------------------------------------------------------------
def collect_competition_probs_targets(model, loader, input_size=(128, 128)):
    model.eval()
    interp = nn.Upsample(size=(input_size[1], input_size[0]), mode='bilinear')

    probs_list, targets_list = [], []

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device)
            logits = interp(model(images))
            probs = F.softmax(logits, dim=1)[:, 1]  # landslide prob

            probs_list.append(probs.cpu().numpy().astype(np.float32))
            targets_list.append(labels.numpy().astype(np.uint8))

    return np.concatenate(probs_list, axis=0), np.concatenate(targets_list, axis=0)


def evaluate_competition_threshold(model, loader, threshold=0.5, input_size=(128, 128)):
    probs, targets = collect_competition_probs_targets(model, loader, input_size)

    preds = (probs >= threshold).astype(np.uint8)
    targets = targets.astype(np.uint8)

    tp = int(((preds == 1) & (targets == 1)).sum())
    fp = int(((preds == 1) & (targets == 0)).sum())
    fn = int(((preds == 0) & (targets == 1)).sum())

    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)

    return {
        "threshold": float(threshold),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "iou": float(iou),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def sweep_competition_threshold(model, val_loader, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.05)

    probs, targets = collect_competition_probs_targets(model, val_loader)

    best = {"f1": -1.0, "threshold": 0.5}

    for t in thresholds:
        preds = (probs >= t).astype(np.uint8)

        tp = int(((preds == 1) & (targets == 1)).sum())
        fp = int(((preds == 1) & (targets == 0)).sum())
        fn = int(((preds == 0) & (targets == 1)).sum())

        prec = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        f1 = 2 * prec * rec / (prec + rec + 1e-8)

        if f1 > best["f1"]:
            best = {
                "threshold": float(t),
                "precision": float(prec),
                "recall": float(rec),
                "f1": float(f1),
            }

    return best
# ------------------------------------------------------------
# 5) Competition-style training loop
#    - Adam
#    - lr=1e-3
#    - wd=5e-4
#    - CE loss
#    - save best by class-1 F1
# ------------------------------------------------------------
def train_competition_unet(
    splits,
    batch_size=8, # to match other experiments
    learning_rate=1e-3,
    weight_decay=5e-4,
    num_steps_stop=5000,
    eval_every=500,
    snapshot_dir=None,
    num_workers=4,
):
    if snapshot_dir is None:
        snapshot_dir = DRIVE_DIR
    os.makedirs(snapshot_dir, exist_ok=True)

    train_dl, val_dl, test_dl = build_competition_dataloaders(
        splits,
        batch_size=batch_size,
        num_workers=num_workers,
    )

    model = CompetitionUNet(n_classes=2, n_channels=14).to(device)
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    interp = nn.Upsample(size=(128, 128), mode='bilinear')
    cross_entropy_loss = nn.CrossEntropyLoss(ignore_index=255)

    best_val_f1 = 0.5
    best_ckpt = None
    history = []

    # Make an "infinite-ish" iterator like their max_iters logic
    train_iter = iter(train_dl)

    for batch_id in range(num_steps_stop):
        t0 = time.time()
        model.train()
        optimizer.zero_grad()

        try:
            images, labels, _ = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl)
            images, labels, _ = next(train_iter)

        images = images.to(device)
        labels = labels.to(device).long()

        pred = model(images)
        pred_interp = interp(pred)

        loss = cross_entropy_loss(pred_interp, labels)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            predict_labels = torch.max(pred_interp, 1)[1]
            batch_oa = (predict_labels == labels).float().mean().item()

        step_time = time.time() - t0
        history.append({
            'step': batch_id + 1,
            'loss': float(loss.item()),
            'batch_oa': float(batch_oa),
            'time': float(step_time),
        })

        if (batch_id + 1) % 10 == 0:
            recent = history[-10:]
            print(
                f"Iter {batch_id+1}/{num_steps_stop} "
                f"Time: {10*np.mean([x['time'] for x in recent]):.2f} "
                f"Batch_OA = {100*np.mean([x['batch_oa'] for x in recent]):.1f} "
                f"cross_entropy_loss = {np.mean([x['loss'] for x in recent]):.3f}"
            )

        if (batch_id + 1) % eval_every == 0:
            print("Validation..........")
            val_metrics = evaluate_competition_threshold(model, val_dl, input_size=(128, 128))

            print(f"===> Landslide Precision: {100*val_metrics['precision']:.2f}")
            print(f"===> Landslide Recall:    {100*val_metrics['recall']:.2f}")
            print(f"===> Landslide F1:        {100*val_metrics['f1']:.2f}")

            if val_metrics["f1"] > best_val_f1:
                best_val_f1 = val_metrics["f1"]
                best_ckpt = os.path.join(
                    snapshot_dir,
                    f"competition_unet_step{batch_id+1}_F1_{int(best_val_f1*10000)}.pth"
                )
                torch.save(model.state_dict(), best_ckpt)
                print(f"Saved best model -> {best_ckpt}")

    if best_ckpt is not None:
        model.load_state_dict(torch.load(best_ckpt, map_location=device))
        print(f"\nLoaded best checkpoint: {best_ckpt}")
        print(f"Best val landslide F1: {best_val_f1:.4f}")

    return model, val_dl, test_dl, best_ckpt

# ------------------------------------------------------------
# 6) Run
# ------------------------------------------------------------
competition_model, competition_val_dl, competition_test_dl, competition_ckpt = train_competition_unet(
    splits=SPLITS,
    batch_size=8,
    learning_rate=1e-3,
    weight_decay=5e-4,
    num_steps_stop=5000,
    eval_every=500,
    snapshot_dir=f"{DRIVE_DIR}/competition_unet_runs",
    num_workers=4,
)

In [ ]:
# @title Validation threshold sweep
best_val = sweep_competition_threshold(
    competition_model,
    competition_val_dl,
)

print("\n=== Competition U-Net Validation Threshold Sweep ===")
print(f"Best threshold : {best_val['threshold']:.2f}")
print(f"Val Precision  : {100*best_val['precision']:.2f}")
print(f"Val Recall     : {100*best_val['recall']:.2f}")
print(f"Val F1         : {100*best_val['f1']:.2f}")

In [ ]:
# @title Test/Evaluation
# Using validation-chosen threshold
test_metrics = evaluate_competition_threshold(
    competition_model,
    competition_test_dl,
    threshold=best_val["threshold"],
)

print("\n=== Final Competition U-Net Test Metrics ===")
print(f"Threshold : {test_metrics['threshold']:.2f}")
print(f"Precision : {100*test_metrics['precision']:.2f}")
print(f"Recall    : {100*test_metrics['recall']:.2f}")
print(f"F1        : {100*test_metrics['f1']:.2f}")
print(f"IoU       : {100*test_metrics['iou']:.2f}")

# Part 4: Architecture 2 (U-net Backbone)

**Design note:** Clay is a ViT AKA it produces one global feature map, not a multi-scale pyramid. Using it as the sole encoder (Arch 1) means the decoder must reconstruct spatial detail from a single coarse representation. Architecture 2 instead uses:

| Component | Detail |
|-----------|--------|
| **U-Net input** | All 14 bands (12 S2 + slope + DEM) |
| **U-Net encoder** | 4 downsample stages: 128→64→32→16→8 (channels: 64→128→256→512→512) |
| **U-Net decoder** | 4 upsample stages: 8→16→32→64→128 (bilinear + skip concat + DoubleConv) |
| **U-Net normalization** | BatchNorm |
| **U-Net skip connections** | Full encoder–decoder skips at all 4 scales |
| **Clay input** | 12 S2 optical bands only (terrain stays in U-Net path) |
| **Clay output** | Single 1024-ch feature map at 16×16 (patch_size=8) |
| **Clay projection** | Conv 1×1 (1024→512) → BatchNorm → ReLU → bilinear resize to 8×8 |
| **Fusion** | Bottleneck-only: concat(U-Net 8×8, Clay 8×8) → two Conv 3×3 → BN → ReLU blocks (1024→512) |
| **Segmentation head** | Conv 1×1 (channels 64→1) at final U-Net output |
| **Decoder dropout** | 10% Dropout2d after each Up block |

In [ ]:
# ============================================================
# @title Hybrid config
# Architecture 2 = U-Net main backbone + Clay auxiliary context
# ============================================================
HYBRID_CFG = {
    # Data
    'input_size': 128,
    'batch_size': 8,   # match Arch 1
    'num_workers': 4,

    # Clay gets Sentinel-2 optical bands only; U-Net gets full 14-band chip.
    's2_bands': list(range(12)),
    'full_bands': list(range(14)),

    # Match Arch 1 training budget.
    'epochs': 50,
    'patience': 10,
    'weight_decay': 1e-3,
    'min_lr': 1e-6,

    # Stage 1: Clay fully frozen, train U-Net + bottleneck fusion.
    # Match Arch 1 decoder-style LR.
    'stage1_lr_unet': 3e-4,
    'stage1_lr_fusion': 3e-4,

    # Stage 2: base Clay still frozen; only Clay LoRA adapters train.
    # Match Arch 1 Stage 2 schedule.
    'stage2_lr_unet': 1e-4,
    'stage2_lr_fusion': 1e-4,
    'stage2_lr_lora': 3e-5, # !changed from 3

    # two-stage LoRA schedule for Arch 2 (True);
    'use_two_stage': True, # adjust based on experiment
    'stage2_start_epoch': 11, # set to 11 if true; else, 999
    'stage2_train_lora': True, # adjust based on experiment

    # clay tuning: no full Clay fine-tuning.
    'use_lora': True,
    'lora_rank': 8,
    'lora_alpha': 8,
    'freeze_clay': True, # Whether clay starts frozen at init
    'unfreeze_last_n_blocks': 0,

    #restrict LoRA scope
    'lora_layers': None,          # possible exp: list(range(20, 24)) == last 4 transformer blocks only
    'lora_target_modules': None, # possible exp: ('to_qkv', 'to_out') ==attention only (no MLP)

    # fusion policy: Clay has one ViT map, so keep Clay bottleneck-only.
    # U-Net itself remains multi-scale through x1..x5 and decoder skips.
    'clay_fusion_level': 'bottleneck_only',

    # Loss
    'loss': 'bce_lovasz',
    'pos_weight': 25.0,
    'bce_weight': 0.90,

    # Fixed-threshold monitoring during training; validation sweep only after training NOT on test
    'monitor_threshold': 0.70, #calibrated based on validation set
    'thresholds': [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85], # help with adjusting thresholds later on

    # Save
    'save_dir': f'{DRIVE_DIR}/hybrid_unet_clay',
    'run_name': 'arch2_unet_clay_bottleneck_twostage_lora_arch1matched_cali', # adjust based on experiment being ran
}
os.makedirs(HYBRID_CFG['save_dir'], exist_ok=True)

In [ ]:
# ============================================================
# @title Stats helpers
# Reuses TRAIN_NORM_STATS from earlier notebook
# ============================================================
def get_train_band_stats():
    s2s  = TRAIN_NORM_STATS['s2']
    ters = TRAIN_NORM_STATS['terrain']
    return {
        **{i: s2s[L4S_BAND_ORDER[i]] for i in range(12)},
        12: ters['slope'],
        13: ters['dem'],
    }

def build_mean_std_arrays(band_indices):
    stats = get_train_band_stats()
    mean = np.array([stats[i]['mean'] for i in band_indices], dtype=np.float32)[:, None, None]
    std  = np.array([stats[i]['std']  for i in band_indices], dtype=np.float32)[:, None, None]
    return mean, std

In [ ]:
# ============================================================
# @title Hybrid dataset
# Returns:
#   x14      -> normalized 14-band input for U-Net
#   s2_x     -> normalized optical subset for Clay
#   mask
# ============================================================
class HybridL4SDataset(Dataset):
    def __init__(self, pairs, cfg, augment=False):
        self.pairs = pairs
        self.cfg = cfg
        self.augment = augment

        self.full_idx = cfg['full_bands']
        self.s2_idx   = cfg['s2_bands']

        self.full_mean, self.full_std = build_mean_std_arrays(self.full_idx)
        self.s2_mean,   self.s2_std   = build_mean_std_arrays(self.s2_idx)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, msk_p = self.pairs[idx]

        with h5py.File(img_p, 'r') as f:
            img = f['img'][:].astype(np.float32)   # (H,W,14)

        with h5py.File(msk_p, 'r') as f:
            key = list(f.keys())[0]
            msk = f[key][:].astype(np.int64)       # (H,W)

        chip = img.transpose(2, 0, 1)              # (14,H,W)

        if self.augment: # apply  augmentation
            chip, msk = augment_geometric(chip, msk)

        # full 14-band branch
        x14_raw = chip[self.full_idx].copy()
        invalid14 = ~np.isfinite(x14_raw)
        if np.any(invalid14):
            x14_raw[invalid14] = np.broadcast_to(self.full_mean, x14_raw.shape)[invalid14]
        x14 = (x14_raw - self.full_mean) / self.full_std

        # optical Clay branch
        s2_raw = chip[self.s2_idx].copy()
        invalid_s2 = ~np.isfinite(s2_raw)
        if np.any(invalid_s2):
            s2_raw[invalid_s2] = np.broadcast_to(self.s2_mean, s2_raw.shape)[invalid_s2]
        s2_x = (s2_raw - self.s2_mean) / self.s2_std

        msk = (msk > 0).astype(np.int64)

        return (
            torch.from_numpy(x14).float(),
            torch.from_numpy(s2_x).float(),
            torch.from_numpy(msk).long(),
        )

def build_hybrid_dataloaders(splits, cfg, num_workers=4):
    train_ds = HybridL4SDataset(splits['train'], cfg, augment=True)
    val_ds   = HybridL4SDataset(splits['validation'], cfg, augment=False)
    test_ds  = HybridL4SDataset(splits['test'], cfg, augment=False)

    dl_common = dict(num_workers=num_workers, pin_memory=True)
    if num_workers > 0:
        dl_common.update(dict(persistent_workers=True, prefetch_factor=4))

    train_dl = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, drop_last=True, **dl_common)
    val_dl   = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, **dl_common)
    test_dl  = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, **dl_common)

    print(f'Dataset sizes → Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    return train_dl, val_dl, test_dl

In [ ]:
# ============================================================
# @title U-Net backbone with bottleneck
# Reuses competition U-Net style
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True, dropout=0.1):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
        self.drop = nn.Dropout2d(p=dropout) #decoder dropout

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.drop(self.conv(x))

class HybridUNetEncoderDecoder(nn.Module):
    def __init__(self, n_channels=14, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, 1, kernel_size=1)

        self.bottleneck_ch = 1024 // factor

    def encode(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        return x1, x2, x3, x4, x5

    def decode(self, x1, x2, x3, x4, x5):
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

In [ ]:
# ============================================================
# @title Clay feature extractor
# Reuses Segmentor + build_clay_datacube(...) from earlier notebook
# ============================================================

class ClayFeatureExtractor(nn.Module):
    """
    Returns ONE final/deep feature map from Clay.

    Clay is ViT-style here, not a U-Net feature pyramid.
    U-Net supplies the spatial pyramid; Clay supplies coarse semantic context.

    Stage 1:
        - Clay has NO LoRA inserted yet
        - Clay is fully frozen

    Stage 2:
        - LoRA adapters are inserted
        - base Clay remains frozen
        - only LoRA params train
    """
    def __init__(self, ckpt_path, s2_bands, cfg):
        super().__init__()

        seg = Segmentor(num_classes=1, ckpt_path=ckpt_path)
        self.encoder = seg.encoder

        # ------------------------------------------------------------
        # IMPORTANT:
        # Do NOT apply LoRA here.
        # If LoRA is inserted before Stage 1, frozen Stage 1 becomes slower
        # and no longer matches the frozen-Clay baseline.
        # ------------------------------------------------------------
        self.use_lora = cfg.get('use_lora', False)
        self.lora_applied = False
        self.lora_rank = cfg.get('lora_rank', 8)
        self.lora_alpha = cfg.get('lora_alpha', 8)

        # Optional: restrict LoRA to later Clay transformer layers / attention only.
        self.lora_layers = cfg.get('lora_layers', None)
        self.lora_target_modules = cfg.get('lora_target_modules', None)

        enc = self.encoder.get_base_model() if hasattr(self.encoder, 'get_base_model') else self.encoder
        self.patch_size = enc.patch_size
        self.embed_dim = 1024
        self.s2_bands = s2_bands

        waves = [L4S_WAVELENGTH[L4S_BAND_ORDER[i]] for i in s2_bands]
        self.register_buffer('clay_waves', torch.tensor(waves, dtype=torch.float32))

        # Stage 1 default: fully frozen Clay.
        self.freeze_all_clay()

    def apply_lora_for_stage2(self):
        """
        Insert LoRA adapters only when Stage 2 begins.

        This keeps Stage 1 equivalent to the original frozen-Clay Arch 2.
        """
        if not self.use_lora:
            print('LoRA disabled; continuing with fully frozen Clay.')
            return

        if self.lora_applied:
            print('LoRA already applied; skipping.')
            return

        print('Applying LoRA adapters to Clay encoder for Stage 2...')

        self.encoder = apply_lora_to_clay_encoder(
            self.encoder,
            rank=self.lora_rank,
            alpha=self.lora_alpha,
            layers=self.lora_layers,
            target_modules=self.lora_target_modules,
        )

        self.lora_applied = True

        # After insertion, freeze base Clay and enable only LoRA params.
        self.set_lora_trainable(True)

    def freeze_all_clay(self):
        """Freeze all Clay params."""
        for p in self.encoder.parameters():
            p.requires_grad = False

    def set_lora_trainable(self, trainable=True):
        """
        Enable only LoRA adapters; base Clay remains frozen.
        Safe to call even before LoRA is applied.
        """
        for name, p in self.encoder.named_parameters():
            if is_lora_param(name):
                p.requires_grad = trainable
            else:
                p.requires_grad = False

    def _get_core(self):
        return self.encoder.get_base_model() if hasattr(self.encoder, 'get_base_model') else self.encoder

    def _tokens_to_map(self, tokens, H_p, W_p):
        expected = H_p * W_p

        if tokens.shape[1] == expected + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected:
            raise ValueError(
                f'Unexpected token count: got {tokens.shape[1]}, '
                f'expected {expected} or {expected + 1}'
            )

        return rearrange(tokens, 'B (H W) D -> B D H W', H=H_p, W=W_p)

    def forward(self, s2_x):
        B, _, H, W = s2_x.shape

        if H % self.patch_size != 0 or W % self.patch_size != 0:
            raise ValueError(
                f'Input size ({H},{W}) not divisible by patch_size={self.patch_size}'
            )

        datacube = build_clay_datacube(
            s2_x,
            self.clay_waves,
            CLAY_TIME.to(s2_x.device),
            CLAY_LATLON.to(s2_x.device),
            CLAY_GSD.to(s2_x.device),
        )

        H_p = H // self.patch_size
        W_p = W // self.patch_size
        enc = self._get_core()

        patches, _ = enc.to_patch_embed(datacube['pixels'], datacube['waves'])
        patches = enc.add_encodings(
            patches,
            datacube['time'],
            datacube['latlon'],
            datacube['gsd'],
        )

        cls = enc.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, patches], dim=1)

        for attn, ff in enc.transformer.layers:
            x = attn(x) + x
            x = ff(x) + x

        return self._tokens_to_map(x, H_p, W_p)

In [ ]:
# ============================================================
# @title Hybrid model
# U-Net main path + Clay bottleneck-only auxiliary fusion
# ============================================================
class HybridUNetClay(nn.Module):
    def __init__(self, cfg, ckpt_path):
        super().__init__()
        self.unet = HybridUNetEncoderDecoder(n_channels=14, bilinear=True)

        self.clay = ClayFeatureExtractor(
            ckpt_path=ckpt_path,
            s2_bands=cfg['s2_bands'],
            cfg=cfg,
        )

        # Project the single Clay ViT map to the U-Net bottleneck channel size.
        self.clay_proj = nn.Sequential(
            nn.Conv2d(self.clay.embed_dim, self.unet.bottleneck_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.unet.bottleneck_ch),
            nn.ReLU(inplace=True),
        )

        # Bottleneck-only fusion. This is intentionally not Clay multiscale fusion.
        self.fuse = nn.Sequential(
            nn.Conv2d(self.unet.bottleneck_ch * 2, self.unet.bottleneck_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(self.unet.bottleneck_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(self.unet.bottleneck_ch, self.unet.bottleneck_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(self.unet.bottleneck_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x14, s2_x):
        # U-Net provides the true multiscale hierarchy.
        x1, x2, x3, x4, x5 = self.unet.encode(x14)

        # Clay provides one coarse semantic context map.
        clay_feat = self.clay(s2_x)
        clay_feat = self.clay_proj(clay_feat)
        clay_feat = F.interpolate(clay_feat, size=x5.shape[-2:], mode='bilinear', align_corners=False)

        x5 = self.fuse(torch.cat([x5, clay_feat], dim=1))
        logits = self.unet.decode(x1, x2, x3, x4, x5)
        return logits

In [ ]:
# ============================================================
# @title Losses (fixed shapes)
# ============================================================
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.cumsum(0)
    union = gts + (1 - gt_sorted).cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    if len(labels) == 0:
        return logits.sum() * 0.
    signs = 2. * labels.float() - 1.
    errors = 1. - logits * signs
    errors_sorted, perm = torch.sort(errors, descending=True)
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)

class HybridBCELovaszLoss(nn.Module):
    def __init__(self, bce_weight=0.90, pos_weight=25.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.register_buffer('pos_weight', torch.tensor([pos_weight], dtype=torch.float32))

    def forward(self, pred, target):
        # pred:   [B,1,H,W]
        # target: [B,1,H,W]
        target = target.float()

        bce = F.binary_cross_entropy_with_logits(
            pred, target, pos_weight=self.pos_weight.view(1, 1, 1, 1)
        )

        pred_flat = pred.squeeze(1)      # [B,H,W]
        target_flat = target.squeeze(1)  # [B,H,W]

        lovasz = torch.stack([
            lovasz_hinge_flat(pred_flat[i].reshape(-1), target_flat[i].reshape(-1))
            for i in range(pred_flat.shape[0])
        ]).mean()

        return self.bce_weight * bce + (1.0 - self.bce_weight) * lovasz

def build_hybrid_loss(cfg):
    if cfg['loss'] == 'bce_lovasz':
        return HybridBCELovaszLoss(
            bce_weight=cfg['bce_weight'],
            pos_weight=cfg['pos_weight']
        ).to(device)
    elif cfg['loss'] == 'bce':
        return nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([cfg['pos_weight']], device=device).view(1, 1, 1, 1)
        )
    else:
        raise ValueError(f"Unknown loss: {cfg['loss']}")

In [ ]:
# ============================================================
# @title Optimizer + two-stage trainability
# ============================================================
def is_lora_param(name):
    name = name.lower()
    return ('lora_' in name) or ('.lora_' in name)


def configure_hybrid_stage_trainability(model, stage, train_lora_stage2=True):
    """
    Stage 1:
      - Freeze Clay completely, including LoRA.
      - Train U-Net + Clay projection/fusion layers.

    Stage 2:
      - Keep base Clay frozen.
      - Enable only Clay LoRA adapters.
      - Continue training U-Net + projection/fusion with lower LR.
    """
    for name, p in model.named_parameters():
        if name.startswith('clay.'):
            p.requires_grad = False
        else:
            p.requires_grad = True

    if stage == 1:
        model.clay.freeze_all_clay()
    elif stage == 2:
        model.clay.set_lora_trainable(trainable=train_lora_stage2)
    else:
        raise ValueError(f'stage must be 1 or 2, got {stage}')


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    clay_trainable = sum(p.numel() for n, p in model.named_parameters() if n.startswith('clay.') and p.requires_grad)
    lora_trainable = sum(p.numel() for n, p in model.named_parameters() if is_lora_param(n) and p.requires_grad)
    print(f'Trainable params: {trainable:,}/{total:,} | Clay trainable: {clay_trainable:,} | LoRA trainable: {lora_trainable:,}')


def build_hybrid_optimizer(model, cfg, stage=1):
    unet_params, fusion_params, lora_params = [], [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('unet.'):
            unet_params.append(p)
        elif name.startswith('clay.') and is_lora_param(name):
            lora_params.append(p)
        else:
            fusion_params.append(p)  # clay_proj + fuse

    if stage == 1:
        lr_unet = cfg['stage1_lr_unet']
        lr_fusion = cfg['stage1_lr_fusion']
        lr_lora = None
        t_max = max(1, cfg.get('stage2_start_epoch', cfg['epochs']) - 1)
    else:
        lr_unet = cfg['stage2_lr_unet']
        lr_fusion = cfg['stage2_lr_fusion']
        lr_lora = cfg['stage2_lr_lora']
        t_max = max(1, cfg['epochs'] - cfg.get('stage2_start_epoch', cfg['epochs']) + 1)

    groups = []
    if unet_params:
        groups.append({'params': unet_params, 'lr': lr_unet})
    if fusion_params:
        groups.append({'params': fusion_params, 'lr': lr_fusion})
    if lora_params and lr_lora is not None:
        groups.append({'params': lora_params, 'lr': lr_lora})

    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max)
    return optimizer, scheduler

In [ ]:
# ============================================================
# @title Metrics + threshold sweep on validation
# ============================================================
def collect_probs_targets(model, loader):
    model.eval()
    all_probs, all_targets = [], []

    with torch.no_grad():
        for x14, s2_x, y in loader:
            x14 = x14.to(device, non_blocking=True)
            s2_x = s2_x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x14, s2_x).squeeze(1)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu())
            all_targets.append(y.cpu())

    return torch.cat(all_probs, dim=0), torch.cat(all_targets, dim=0)

def compute_f1_at_threshold(probs, targets, t):
    preds = (probs >= t).long()
    y = targets.long()

    tp = ((preds == 1) & (y == 1)).sum().item()
    fp = ((preds == 1) & (y == 0)).sum().item()
    fn = ((preds == 0) & (y == 1)).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)

    return {
        'threshold': t,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
        'tp': tp, 'fp': fp, 'fn': fn,
    }

def sweep_thresholds(model, loader, thresholds):
    probs, targets = collect_probs_targets(model, loader)
    results = [compute_f1_at_threshold(probs, targets, t) for t in thresholds]
    best = max(results, key=lambda x: x['f1'])
    return best, results

In [ ]:
# ============================================================
# Missing Clay helper for hybrid model
# Run this before train_hybrid_model(...)
# ============================================================

# If not already defined earlier
if 'CLAY_TIME' not in globals():
    CLAY_TIME = torch.tensor([1., 1., 1., 1.], dtype=torch.float32)
if 'CLAY_LATLON' not in globals():
    CLAY_LATLON = torch.tensor([0., 0., 0., 0.], dtype=torch.float32)
if 'CLAY_GSD' not in globals():
    CLAY_GSD = torch.tensor(10.0, dtype=torch.float32)

def build_clay_datacube(s2_x, waves, clay_time, clay_latlon, clay_gsd):
    """
    Build the dict Clay encoder expects.

    Args:
        s2_x:        [B, C, H, W]
        waves:       [C]
        clay_time:   [4]
        clay_latlon: [4]
        clay_gsd:    scalar tensor
    """
    B = s2_x.shape[0]
    return {
        "pixels": s2_x,
        "time": clay_time.expand(B, -1),
        "latlon": clay_latlon.expand(B, -1),
        "gsd": clay_gsd,
        "waves": waves,
    }

In [ ]:
# ============================================================
# @title Train loop
# - Two-stage Arch 2 training
# - Fixed-threshold validation during training
# - Validation threshold sweep only AFTER training
# ============================================================
def run_hybrid_epoch(model, loader, criterion, optimizer=None, scaler=None, phase='train'):
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()

    total_loss = 0.0
    use_amp = (device.type == 'cuda')

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x14, s2_x, y in loader:
            x14 = x14.to(device, non_blocking=True)
            s2_x = s2_x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                logits = model(x14, s2_x)
                loss = criterion(logits, y.unsqueeze(1))

            if is_train:
                if scaler is not None and use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)


def train_hybrid_model(cfg):
    train_dl, val_dl, test_dl = build_hybrid_dataloaders(SPLITS, cfg, num_workers=cfg['num_workers'])

    model = HybridUNetClay(cfg, CKPT_PATH).to(device)
    criterion = build_hybrid_loss(cfg)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    use_two_stage = cfg.get('use_two_stage', True)
    stage2_start = int(cfg.get('stage2_start_epoch', cfg['epochs'] + 1))

    current_stage = 1 if use_two_stage else 2
    configure_hybrid_stage_trainability(model, stage=current_stage, train_lora_stage2=cfg.get('stage2_train_lora', True))
    optimizer, scheduler = build_hybrid_optimizer(model, cfg, stage=current_stage)

    print(f"\n=== Training {cfg['run_name']} ===")
    print('Fusion: Clay bottleneck-only auxiliary context; U-Net provides multi-scale spatial path.')
    print(f'Stage {current_stage} initialized')
    count_trainable_params(model)

    best_val_f1 = -1.0
    best_epoch = 0
    patience_count = 0
    best_ckpt = f"{cfg['save_dir']}/{cfg['run_name']}_best.pt"
    monitor_t = cfg.get('monitor_threshold', 0.5)
    history = []

    print(f'{"Epoch":>5} {"Stage":>5} {"TrLoss":>8} {"VaLoss":>8} {"VaF1@t":>8} {"Th":>5} {"VaP":>7} {"VaR":>7} {"VaIoU":>7} {"Time":>6}')
    print('-' * 82)

    for epoch in range(1, cfg['epochs'] + 1):
        if use_two_stage and epoch == stage2_start:
            current_stage = 2

            print(f'\n--- Switching to Stage 2 at epoch {epoch}: base Clay frozen, LoRA-only Clay trainable ---')

            # 1. Insert LoRA adapters into Clay NOW (not at __init__).
            model.clay.apply_lora_for_stage2()

            # 2. Freeze base Clay, enable only LoRA params.
            configure_hybrid_stage_trainability(
                model, stage=2,
                train_lora_stage2=cfg.get('stage2_train_lora', True),
            )

            # 3. Rebuild optimizer AFTER LoRA params exist.
            optimizer, scheduler = build_hybrid_optimizer(model, cfg, stage=2)

            # Reset patience so Stage 2 gets a full runway.
            patience_count = 0

            count_trainable_params(model)

        t0 = time.time()
        train_loss = run_hybrid_epoch(model, train_dl, criterion, optimizer=optimizer, scaler=scaler, phase='train')
        val_loss   = run_hybrid_epoch(model, val_dl, criterion, optimizer=None, scaler=None, phase='val')

        # Checkpoint by fixed threshold only; sweep comes after training.
        probs, targets = collect_probs_targets(model, val_dl)
        val_fixed = compute_f1_at_threshold(probs, targets, monitor_t)
        scheduler.step()

        improved = val_fixed['f1'] > best_val_f1
        if improved:
            best_val_f1 = val_fixed['f1']
            best_epoch = epoch
            patience_count = 0
            local_ckpt = f'/content/{cfg["run_name"]}_best.pt'
            torch.save(model.state_dict(), local_ckpt)
            import shutil
            shutil.copy2(local_ckpt, best_ckpt)
        else:
            patience_count += 1

        history.append({
            'epoch': epoch,
            'stage': current_stage,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_f1_fixed': val_fixed['f1'],
            'val_precision_fixed': val_fixed['precision'],
            'val_recall_fixed': val_fixed['recall'],
            'val_iou_fixed': val_fixed['iou'],
            'monitor_threshold': monitor_t,
        })

        flag = ' *' if improved else ''
        print(f"{epoch:>5} {current_stage:>5} {train_loss:>8.4f} {val_loss:>8.4f} {val_fixed['f1']:>8.4f} {monitor_t:>5.2f} "
              f"{val_fixed['precision']:>7.4f} {val_fixed['recall']:>7.4f} {val_fixed['iou']:>7.4f} {time.time()-t0:>6.0f}s{flag}")

        if patience_count >= cfg['patience']:
            print(f'Early stopping at epoch {epoch}')
            break

    # Load best fixed-threshold checkpoint, then sweep threshold ONCE on validation.
    local_ckpt = f'/content/{cfg["run_name"]}_best.pt'

    if use_two_stage and best_epoch >= stage2_start:
        model.clay.apply_lora_for_stage2()
    elif model.clay.lora_applied:
        model = HybridUNetClay(cfg, CKPT_PATH).to(device)

    model.load_state_dict(torch.load(local_ckpt, map_location=device))
    best_val_swept, sweep_results = sweep_thresholds(model, val_dl, cfg['thresholds'])
    best_thresh = best_val_swept['threshold']

    print(f"\nBest checkpoint by fixed val F1@{monitor_t:.2f}: epoch {best_epoch}, F1={best_val_f1:.4f}")
    print(f"Validation-only post-training sweep selected threshold={best_thresh:.2f}, swept val F1={best_val_swept['f1']:.4f}")

    return model, train_dl, val_dl, test_dl, history, best_thresh, sweep_results

In [ ]:
# ============================================================
# @title Run Train + Test/Evaluation
# - Threshold selected on validation only AFTER training
# - Test evaluated once using the validation-selected threshold
# ============================================================
def evaluate_locked_threshold(model, loader, threshold):
    probs, targets = collect_probs_targets(model, loader)
    return compute_f1_at_threshold(probs, targets, threshold)

hybrid_model, hybrid_train_dl, hybrid_val_dl, hybrid_test_dl, hybrid_history, hybrid_best_thresh, hybrid_sweep_results = train_hybrid_model(HYBRID_CFG)

val_final  = evaluate_locked_threshold(hybrid_model, hybrid_val_dl, hybrid_best_thresh)
test_final = evaluate_locked_threshold(hybrid_model, hybrid_test_dl, hybrid_best_thresh)

print("\n=== Final Locked-Threshold Metrics ===")
print(f"Validation -> F1: {val_final['f1']:.4f} | P: {val_final['precision']:.4f} | R: {val_final['recall']:.4f} | IoU: {val_final['iou']:.4f} | t={hybrid_best_thresh:.2f}")
print(f"Test       -> F1: {test_final['f1']:.4f} | P: {test_final['precision']:.4f} | R: {test_final['recall']:.4f} | IoU: {test_final['iou']:.4f} | t={hybrid_best_thresh:.2f}")

In [ ]:
# ============================================================
# @title t=0.50 comparison + ROC curve
# Adds to existing validation sweep NOT re-sweep
# ============================================================
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Single inference pass on test set
test_probs_t, test_targets_t = collect_probs_targets(hybrid_model, hybrid_test_dl)
test_probs_np   = test_probs_t.numpy().ravel()
test_targets_np = test_targets_t.numpy().ravel().astype(np.uint8)

# --- 1. Locked vs t=0.50 comparison ---
m_locked = compute_f1_at_threshold(test_probs_t, test_targets_t, hybrid_best_thresh)
m_050    = compute_f1_at_threshold(test_probs_t, test_targets_t, 0.50)

print(f"=== Threshold comparison (Arch 2 LoRA two-stage) ===")
print(f"t={hybrid_best_thresh:.2f} (locked): P={m_locked['precision']:.4f}  R={m_locked['recall']:.4f}  F1={m_locked['f1']:.4f}")
print(f"t=0.50            : P={m_050['precision']:.4f}  R={m_050['recall']:.4f}  F1={m_050['f1']:.4f}")
print(f"Delta (locked - 0.50): dP={m_locked['precision']-m_050['precision']:+.4f}  "
      f"dR={m_locked['recall']-m_050['recall']:+.4f}  dF1={m_locked['f1']-m_050['f1']:+.4f}")

# --- 2. ROC curve ---
fpr, tpr, _ = roc_curve(test_targets_np, test_probs_np)
roc_auc = auc(fpr, tpr)
# Compute PR-AUC as a number for the JSON (no plot)
pr_auc = average_precision_score(test_targets_np, test_probs_np)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f'Arch 2 (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — Test Set (AUC={roc_auc:.3f})')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()

local_path = f'/content/{HYBRID_CFG["run_name"]}_roc.png'
fig.savefig(local_path, dpi=150, bbox_inches='tight')
drive_path = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_roc.png'
shutil.copy2(local_path, drive_path)
plt.show()
print(f'Saved: {drive_path}')

# Save numerical results for the paper
threshold_results = {
    'run_name': HYBRID_CFG['run_name'],
    'locked_threshold': float(hybrid_best_thresh),
    'roc_auc': float(roc_auc),
    'pr_auc': float(pr_auc),
    'metrics_at_locked': {k: float(v) for k, v in m_locked.items() if k != 'threshold'},
    'metrics_at_0.50':   {k: float(v) for k, v in m_050.items() if k != 'threshold'},
}
with open(f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_roc_results.json', 'w') as f:
    json.dump(threshold_results, f, indent=2)
print('Saved roc_results.json')

In [ ]:
# ============================================================
# @title Multi-seed: Model 7
# Per-restart: change SEEDS_A2 to [42], then [123], then [2024]
# Final aggregate runs ONLY when all 3 seeds present.
# ============================================================
#Define helpers locally so this Part is self-contained (doesn't depend on Part 2 being run)
def set_all_seeds(seed):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def evaluate_locked_threshold(model, loader, threshold):
    probs, targets = collect_probs_targets(model, loader)
    return compute_f1_at_threshold(probs, targets, threshold)

SEEDS_A2 = [2024]   # ← CHANGE PER RESTART: [42], [123], [2024]
ALL_SEEDS = [42, 123, 2024]   # full set expected when done

partial_path = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_multiseed_partial.json'

# Resume if partial exists
if os.path.exists(partial_path):
    with open(partial_path) as f:
        saved = json.load(f)
    multi_seed_a2 = saved['results']
    done_seeds = {r['seed'] for r in multi_seed_a2}
    print(f'Resuming: already have seeds {sorted(done_seeds)}')
else:
    multi_seed_a2 = []
    done_seeds = set()

for seed in SEEDS_A2:
    if seed in done_seeds:
        print(f'Skip seed {seed} (done)'); continue
    print(f'\n{"="*60}\nARCH 2 — SEED {seed}\n{"="*60}')
    set_all_seeds(seed)

    seed_cfg = copy.deepcopy(HYBRID_CFG)
    seed_cfg['run_name'] = f'{HYBRID_CFG["run_name"]}_seed{seed}'

    model_s, _, val_dl_s, test_dl_s, _, best_t_s, _ = train_hybrid_model(seed_cfg)

    test_locked = evaluate_locked_threshold(model_s, test_dl_s, best_t_s)
    probs_s, tgts_s = collect_probs_targets(model_s, test_dl_s)
    test_050 = compute_f1_at_threshold(probs_s, tgts_s, 0.50)

    multi_seed_a2.append({
        'seed': seed, 'best_threshold': float(best_t_s),
        'test_precision': float(test_locked['precision']),
        'test_recall':    float(test_locked['recall']),
        'test_f1':        float(test_locked['f1']),
        'test_precision_t050': float(test_050['precision']),
        'test_recall_t050':    float(test_050['recall']),
        'test_f1_t050':        float(test_050['f1']),
    })
    print(f"Seed {seed}: F1@locked(t={best_t_s:.2f})={test_locked['f1']:.4f}  |  F1@0.50={test_050['f1']:.4f}")

    # Incremental save
    with open(partial_path, 'w') as f:
        json.dump({'results': multi_seed_a2}, f, indent=2)

    del model_s, val_dl_s, test_dl_s, probs_s, tgts_s
    torch.cuda.empty_cache()
    import gc; gc.collect()

In [ ]:
# ----------------------------------------------------------
# Aggregate ONLY when all expected seeds are complete
# ----------------------------------------------------------
completed_seeds = {r['seed'] for r in multi_seed_a2}
if set(ALL_SEEDS).issubset(completed_seeds):
    def ms(vals): a=np.array(vals); return float(a.mean()), float(a.std(ddof=1)) if len(a)>1 else 0.0
    agg = {k: dict(zip(['mean','std'], ms([r[k] for r in multi_seed_a2])))
           for k in ['test_precision','test_recall','test_f1',
                     'test_precision_t050','test_recall_t050','test_f1_t050','best_threshold']}

    print(f'\n=== ARCH 2 Multi-seed Summary (n={len(multi_seed_a2)}) ===')
    print(f"Locked t (mean={agg['best_threshold']['mean']:.2f} ± {agg['best_threshold']['std']:.2f}):")
    print(f"  P  = {agg['test_precision']['mean']*100:.1f} ± {agg['test_precision']['std']*100:.1f}")
    print(f"  R  = {agg['test_recall']['mean']*100:.1f} ± {agg['test_recall']['std']*100:.1f}")
    print(f"  F1 = {agg['test_f1']['mean']*100:.1f} ± {agg['test_f1']['std']*100:.1f}")
    print(f"At t=0.50:")
    print(f"  P  = {agg['test_precision_t050']['mean']*100:.1f} ± {agg['test_precision_t050']['std']*100:.1f}")
    print(f"  R  = {agg['test_recall_t050']['mean']*100:.1f} ± {agg['test_recall_t050']['std']*100:.1f}")
    print(f"  F1 = {agg['test_f1_t050']['mean']*100:.1f} ± {agg['test_f1_t050']['std']*100:.1f}")

    with open(f'{HYBRID_CFG["save_dir"]}/arch2_multiseed_final.json','w') as f:
        json.dump({'seeds': sorted(completed_seeds),
                   'per_seed': multi_seed_a2, 'aggregate': agg}, f, indent=2)
    print('Saved arch2_multiseed_final.json')
else:
    remaining = sorted(set(ALL_SEEDS) - completed_seeds)
    print(f'\n{"="*60}')
    print(f'Completed seeds: {sorted(completed_seeds)}')
    print(f'Remaining seeds: {remaining}')
    print(f'Aggregate skipped — will compute once all seeds finish.')
    print(f'Next: restart runtime, set SEEDS_A2 = [{remaining[0]}], rerun this cell.')
    print(f'{"="*60}')

In [ ]:
# ============================================================
# @title Load seed 42 model for visualization figures
# ============================================================
SEED42_NAME = f'{HYBRID_CFG["run_name"]}_seed42'
seed42_ckpt = f'{HYBRID_CFG["save_dir"]}/{SEED42_NAME}_best.pt'

assert os.path.exists(seed42_ckpt), f"Seed 42 checkpoint not found: {seed42_ckpt}"

# Rebuild architecture and dataloaders
set_all_seeds(42)
seed42_cfg = copy.deepcopy(HYBRID_CFG)
seed42_cfg['run_name'] = SEED42_NAME

hybrid_train_dl, hybrid_val_dl, hybrid_test_dl = build_hybrid_dataloaders(
    SPLITS, seed42_cfg, num_workers=seed42_cfg['num_workers']
)

# Build model and apply LoRA so state_dict shapes match the saved checkpoint
hybrid_model = HybridUNetClay(seed42_cfg, CKPT_PATH).to(device)
hybrid_model.clay.apply_lora_for_stage2()   # checkpoint was saved post-stage-2

# Load saved weights
state = torch.load(seed42_ckpt, map_location=device)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
hybrid_model.load_state_dict(state, strict=True)
hybrid_model.eval()

# Recover seed 42's locked threshold from the multi-seed JSON
partial_path = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_multiseed_partial.json'
with open(partial_path) as f:
    multiseed = json.load(f)['results']
seed42_record = next(r for r in multiseed if r['seed'] == 42)
hybrid_best_thresh = seed42_record['best_threshold']

# Recompute test_final at locked threshold
test_final = evaluate_locked_threshold(hybrid_model, hybrid_test_dl, hybrid_best_thresh)

print(f"Loaded seed 42 model")
print(f"  Locked threshold: {hybrid_best_thresh}")
print(f"  Test F1: {test_final['f1']:.4f}")
print(f"  Test P:  {test_final['precision']:.4f}")
print(f"  Test R:  {test_final['recall']:.4f}")

In [ ]:
# ============================================================
# @title Qualitative visualization + confusion matrix (TEST set)
# ============================================================

hybrid_model.eval()

# Single pass: collect all probs, targets, and a few x14 for RGB
N_HIGH = 4
N_SPARSE = 2
SPARSE_RANGE = (0.01, 0.05)

all_probs = []
all_targets = []
rgb_candidates = []  # store (pos_ratio, x14_cpu, mask_cpu, prob_cpu)

with torch.no_grad():
    for x14, s2_x, y in hybrid_test_dl:
        x14 = x14.to(device, non_blocking=True)
        s2_x = s2_x.to(device, non_blocking=True)

        logits = hybrid_model(x14, s2_x).squeeze(1)
        probs = torch.sigmoid(logits).cpu()

        all_probs.append(probs)
        all_targets.append(y)

        # Only keep x14 on CPU for samples worth visualizing
        for i in range(x14.shape[0]):
            pos_ratio = y[i].float().mean().item()
            if pos_ratio >= 0.01:
                rgb_candidates.append((pos_ratio, x14[i].cpu(), y[i].cpu(), probs[i]))

all_probs = torch.cat(all_probs, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy().astype(np.uint8)
all_preds = (all_probs >= hybrid_best_thresh).astype(np.uint8)

# Pixel-level confusion matrix (full test set)
tp = int(((all_preds == 1) & (all_targets == 1)).sum())
fp = int(((all_preds == 1) & (all_targets == 0)).sum())
fn = int(((all_preds == 0) & (all_targets == 1)).sum())
tn = int(((all_preds == 0) & (all_targets == 0)).sum())

cm = np.array([[tn, fp], [fn, tp]])

fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
im = ax_cm.imshow(cm, cmap='Blues')

labels = ['Background', 'Landslide']
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() * 0.5 else 'black'
        ax_cm.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                   fontsize=13, fontweight='bold', color=color)

ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(labels); ax_cm.set_yticklabels(labels)
ax_cm.set_xlabel('Predicted', fontsize=11)
ax_cm.set_ylabel('Actual', fontsize=11)
ax_cm.set_title(
    f'Arch 2 Pixel Confusion Matrix(t={hybrid_best_thresh:.2f})\n'
    f'F1={test_final["f1"]:.4f} | P={test_final["precision"]:.4f} | R={test_final["recall"]:.4f}',
    fontsize=11
)
plt.colorbar(im, ax=ax_cm, fraction=0.046)
plt.tight_layout()

local_cm_path = f'/content/{HYBRID_CFG["run_name"]}_confusion_matrix.png'
fig_cm.savefig(local_cm_path, dpi=150, bbox_inches='tight')
import shutil
drive_cm_path = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_confusion_matrix.png'
shutil.copy2(local_cm_path, drive_cm_path)
plt.show()
print(f'Saved: {drive_cm_path}')

# Select high-pos and sparse samples for qualitative vis
rgb_candidates.sort(key=lambda c: c[0], reverse=True)
high = rgb_candidates[:N_HIGH]
sparse = [c for c in rgb_candidates if SPARSE_RANGE[0] <= c[0] <= SPARSE_RANGE[1]][:N_SPARSE]
chosen = high + sparse

def denorm_rgb(x14_tensor):
    stats = get_train_band_stats()
    idxs = [3, 2, 1]  # B4, B3, B2
    rgb = []
    for i in idxs:
        ch = x14_tensor[i].numpy() * stats[i]['std'] + stats[i]['mean']
        rgb.append(ch)
    rgb = np.stack(rgb, axis=-1)
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)

n_show = len(chosen)
thresh = hybrid_best_thresh

fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for i, (pos_ratio, x14_i, mask_i, prob_i) in enumerate(chosen):
    rgb  = denorm_rgb(x14_i)
    mask = mask_i.numpy().astype(np.uint8)
    prob = prob_i.numpy()
    pred = (prob >= thresh).astype(np.uint8)

    overlay = rgb.copy()
    overlay[(pred == 1) & (mask == 1)] = [0.0, 0.8, 0.0]
    overlay[(pred == 1) & (mask == 0)] = [0.9, 0.0, 0.0]
    overlay[(pred == 0) & (mask == 1)] = [1.0, 0.5, 0.0]

    group = 'High-pos' if i < len(high) else 'Sparse'
    panels = [
        (rgb, {}),
        (mask, {'cmap': 'gray', 'vmin': 0, 'vmax': 1}),
        ((pred * 255).astype(np.uint8), {'cmap': 'gray', 'vmin': 0, 'vmax': 255}),
        (overlay, {}),
    ]
    titles = [
        f'{group} RGB\npos={pos_ratio:.2%}',
        'Ground Truth',
        f'Prediction\nt={thresh:.2f}',
        'TP/FP/FN Overlay',
    ]
    for j, (data, kw) in enumerate(panels):
        axes[i, j].imshow(data, **kw)
        axes[i, j].set_title(titles[j], fontsize=11)
        axes[i, j].axis('off')

fig.legend(
    handles=[
        mpatches.Patch(color='#00cc00', label='TP'),
        mpatches.Patch(color='#cc0000', label='FP'),
        mpatches.Patch(color='#ff8000', label='FN'),
    ],
    loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.01)
)
plt.suptitle(
    f'Arch 2: U-Net + Clay bottleneck + two-stage LoRA (F1={test_final["f1"]:.3f})',
    fontsize=12
)
plt.tight_layout()

local_vis_path = f'/content/{HYBRID_CFG["run_name"]}_test_predictions.png'
fig.savefig(local_vis_path, dpi=150, bbox_inches='tight')
drive_vis_path = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}_test_predictions.png'
shutil.copy2(local_vis_path, drive_vis_path)
plt.show()
print(f'Saved: {drive_vis_path}')

In [ ]:
# ============================================================
# Dirichlet calibration (binary form: p = sigmoid(a*z + b))
# Generalizes temperature scaling with a learnable bias term
# ============================================================
from scipy.optimize import minimize

hybrid_model.eval()
cal_logits, cal_targets = [], []
with torch.no_grad():
    for x14, s2_x, y in hybrid_val_dl:
        x14 = x14.to(device, non_blocking=True)
        s2_x = s2_x.to(device, non_blocking=True)
        logits = hybrid_model(x14, s2_x).squeeze(1)
        cal_logits.append(logits.cpu())
        cal_targets.append(y.float())

cal_logits = torch.cat(cal_logits, dim=0)
cal_targets = torch.cat(cal_targets, dim=0)

def dir_loss(params):
    a, b = params
    return F.binary_cross_entropy(
        torch.sigmoid(a * cal_logits + b), cal_targets, reduction='mean'
    ).item()

# Cold start from identity (a=1, b=0 = no calibration)
result = minimize(dir_loss, x0=[1.0, 0.0], method='Nelder-Mead',
                  options={'xatol': 1e-6, 'fatol': 1e-6})
best_a, best_b = result.x
loss_dir = dir_loss(result.x)
loss_identity = dir_loss([1.0, 0.0])  # uncalibrated baseline for reference

hybrid_model.cal_a = best_a
hybrid_model.cal_b = best_b

print(f'Dirichlet calibration:  a={best_a:.4f}, b={best_b:.4f}')
print(f'Validation BCE: {loss_identity:.4f} (uncalibrated) → {loss_dir:.4f} (calibrated)')
print(f'Relative reduction: {(1 - loss_dir/loss_identity)*100:.1f}%')

In [ ]:
# ============================================================
# @title MC Dropout Uncertainty Quantification
# Keep Dropout2d active at inference, run T forward passes,
# variance across passes = epistemic uncertainty map.
# ============================================================
T = 20

# Enable dropout at inference (BatchNorm stays eval)
hybrid_model.eval()
dropout_count = 0
for m in hybrid_model.modules():
    if isinstance(m, (nn.Dropout, nn.Dropout2d)):
        m.train()
        dropout_count += 1
print(f'MC Dropout: {dropout_count} dropout layers active for inference')

# T stochastic forward passes
print(f'Running {T} forward passes...')
t0 = time.time()

all_passes = []
all_targets = []

with torch.no_grad():
    for t_pass in range(T):
        pass_probs = []
        for x14, s2_x, y in hybrid_test_dl:
            x14  = x14.to(device, non_blocking=True)
            s2_x = s2_x.to(device, non_blocking=True)
            logits = hybrid_model(x14, s2_x).squeeze(1)
            a = hybrid_model.cal_a
            b = hybrid_model.cal_b
            pass_probs.append(torch.sigmoid(a * logits + b).cpu().numpy())
            if t_pass == 0:
                all_targets.append(y.numpy())
        all_passes.append(np.concatenate(pass_probs, axis=0))

all_passes  = np.stack(all_passes, axis=0)   # (T, N, H, W)
all_targets = np.concatenate(all_targets, axis=0)
print(f'Done in {time.time()-t0:.0f}s')

# Uncertainty maps
mean_probs  = all_passes.mean(axis=0)         # (N, H, W)
uncertainty = all_passes.var(axis=0)           # (N, H, W) — epistemic
std_map     = np.sqrt(uncertainty)

# Metrics by confusion category
preds   = (mean_probs >= hybrid_best_thresh).astype(np.uint8)
tgt     = all_targets.astype(np.uint8)
tp_mask = (preds == 1) & (tgt == 1)
fp_mask = (preds == 1) & (tgt == 0)
fn_mask = (preds == 0) & (tgt == 1)
tn_mask = (preds == 0) & (tgt == 0)

safe = lambda arr, mask: float(arr[mask].mean()) if mask.any() else 0.0

print(f'\n--- MC Dropout Uncertainty (T={T}) ---')
print(f'Mean uncertainty (all):  {std_map.mean():.6f}')
print(f'Mean uncertainty (TP):   {safe(std_map, tp_mask):.6f}')
print(f'Mean uncertainty (FP):   {safe(std_map, fp_mask):.6f}')
print(f'Mean uncertainty (FN):   {safe(std_map, fn_mask):.6f}')
print(f'Mean uncertainty (TN):   {safe(std_map, tn_mask):.6f}')
print(f'Frac pixels σ > 0.01:   {(std_map > 0.01).mean():.4f}')
print(f'Frac pixels σ > 0.05:   {(std_map > 0.05).mean():.4f}')

# Visualize selected samples
pos_ratios = all_targets.mean(axis=(1, 2))
high_idx   = np.argsort(pos_ratios)[-4:][::-1].tolist()
sparse_idx = [i for i in range(len(pos_ratios)) if 0.01 < pos_ratios[i] < 0.05][:2]
plot_idx   = high_idx + sparse_idx

n = len(plot_idx)
fig, axes = plt.subplots(n, 4, figsize=(18, 4.2 * n))
if n == 1:
    axes = axes[np.newaxis, :]

std_max = max(std_map[plot_idx].max(), 1e-6)

for row, idx in enumerate(plot_idx):
    gt   = all_targets[idx]
    mp   = mean_probs[idx]
    sd   = std_map[idx]
    pred = (mp >= hybrid_best_thresh).astype(np.uint8)

    axes[row, 0].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'GT (pos={pos_ratios[idx]:.2%})')

    axes[row, 1].imshow(mp, cmap='RdYlGn', vmin=0, vmax=1)
    axes[row, 1].set_title(f'MC Mean Prob (t={hybrid_best_thresh:.2f})')

    im = axes[row, 2].imshow(sd, cmap='hot', vmin=0, vmax=std_max)
    axes[row, 2].set_title('Uncertainty (σ)')
    plt.colorbar(im, ax=axes[row, 2], fraction=0.046)

    overlay = np.zeros((*gt.shape, 3))
    overlay[(pred == 1) & (gt == 1)] = [0.0, 0.8, 0.0]
    overlay[(pred == 1) & (gt == 0)] = [0.9, 0.0, 0.0]
    overlay[(pred == 0) & (gt == 1)] = [1.0, 0.5, 0.0]
    ep_thresh = max(np.percentile(sd, 95), 0.01)
    overlay[sd > ep_thresh] = [0.2, 0.4, 1.0]
    axes[row, 3].imshow(overlay)
    axes[row, 3].set_title('TP/FP/FN + High-σ (blue)')

    for j in range(4):
        axes[row, j].axis('off')

fig.legend(
    handles=[
        mpatches.Patch(color='#00cc00', label='TP'),
        mpatches.Patch(color='#cc0000', label='FP'),
        mpatches.Patch(color='#ff8000', label='FN'),
        mpatches.Patch(color='#3366ff', label='High uncertainty'),
    ],
    loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.01)
)
plt.suptitle(
    f'{HYBRID_CFG["run_name"]} — MC Dropout Uncertainty (T={T})', fontsize=13
)
plt.tight_layout()

local_path = f'/content/{HYBRID_CFG["run_name"]}_mc_dropout.png'
fig.savefig(local_path, dpi=150, bbox_inches='tight')
drive_dir = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy2(local_path, f'{drive_dir}/mc_dropout_uncertainty.png')
plt.show()

# Save arrays + metrics
np.savez_compressed(f'{drive_dir}/mc_dropout_arrays.npz',
                    mean=mean_probs, std=std_map, targets=all_targets)
uq_metrics = {
    'run_name': HYBRID_CFG['run_name'], 'T': T,
    'threshold': float(hybrid_best_thresh),
    'mean_std_all': float(std_map.mean()),
    'mean_std_tp': safe(std_map, tp_mask),
    'mean_std_fp': safe(std_map, fp_mask),
    'mean_std_fn': safe(std_map, fn_mask),
    'mean_std_tn': safe(std_map, tn_mask),
}
with open(f'{drive_dir}/mc_dropout_metrics.json', 'w') as f:
    json.dump(uq_metrics, f, indent=2)

hybrid_model.eval()  # restore full eval mode
print(f'\nSaved to {drive_dir}/')

In [ ]:
# ============================================================
# @title Grad-CAM: fusion vs pre-fusion (signed) + Δ
#   Bottleneck (pre-fusion, 8×8) | Fusion (8×8) | Δ(fusion − pre-fusion)
# Signed CAM (NO ReLU) + diverging colormap. Stratified chip sampling.
# Test-set Δ aggregate printed at end.
# ============================================================
import random

# ---- KNOBS -------------------------------------------------
RNG_SEED   = 7        # change this int for a fresh set of example chips
K_PER_BIN  = 1        # rows per positive-ratio bin
BINS = [              # (low, high) positive-ratio ranges
    (0.70, 1.00),   # near-saturated
    (0.30, 0.70),   # dense
    (0.10, 0.30),   # moderate
    (0.02, 0.10),   # sparse
    (0.00, 0.02),   # near-empty
]


class GradCAMHook:
    def __init__(self, module):
        self.activations = None; self.gradients = None
        self._fwd = module.register_forward_hook(self._save_activation)
        self._bwd = module.register_full_backward_hook(self._save_gradient)
    def _save_activation(self, m, i, o): self.activations = o.detach()
    def _save_gradient(self, m, gi, go): self.gradients = go[0].detach()
    def remove(self): self._fwd.remove(); self._bwd.remove()


def compute_gradcam_signed(model, x14, s2_x, target_layer, target_size):
    """Signed Grad-CAM (no ReLU). Red = positive, blue = negative contribution."""
    hook = GradCAMHook(target_layer)
    inplace_relus = []
    for m in model.modules():
        if isinstance(m, nn.ReLU) and m.inplace:
            m.inplace = False; inplace_relus.append(m)
    model.eval()
    score = torch.sigmoid(model(x14, s2_x)).mean()
    model.zero_grad(); score.backward()
    weights = hook.gradients.mean(dim=(2, 3), keepdim=True)
    cam = (weights * hook.activations).sum(dim=1, keepdim=True)   # NO relu -> signed
    cam = F.interpolate(cam, size=target_size, mode='bilinear', align_corners=False)
    cam = cam.squeeze().cpu().numpy()
    hook.remove()
    for m in inplace_relus: m.inplace = True
    return cam


def norm_signed(a):
    m = np.abs(a).max()
    return a / m if m > 1e-8 else np.zeros_like(a)


# 1. Collect all test-set candidates
hybrid_model.eval()
candidates = []
with torch.no_grad():
    for x14, s2_x, y in hybrid_test_dl:
        x14, s2_x = x14.to(device), s2_x.to(device)
        probs = torch.sigmoid(hybrid_model(x14, s2_x).squeeze(1)).cpu()
        for i in range(x14.shape[0]):
            candidates.append({'x14': x14[i:i+1], 's2_x': s2_x[i:i+1],
                               'mask': y[i].cpu().numpy(), 'prob': probs[i].numpy(),
                               'pos_ratio': y[i].float().mean().item()})

# 2. Stratified sampling across positive-ratio bins
rng = random.Random(RNG_SEED)
chosen = []
for lo, hi in BINS:
    pool = [c for c in candidates if lo <= c['pos_ratio'] < hi]
    rng.shuffle(pool)
    chosen.extend(pool[:K_PER_BIN])

print(f"Selected {len(chosen)} chips across positive-ratio bins:")
for c in chosen:
    print(f"  pos={c['pos_ratio']:.2%}")
if len(chosen) == 0:
    raise RuntimeError("No chips selected — check BINS ranges against your test set.")

# 3. Compute Grad-CAMs for selected chips
layer_pre  = hybrid_model.unet.down4.maxpool_conv[1].double_conv[-1]  # pre-fusion, 8×8
layer_fuse = hybrid_model.fuse[-1]                                    # fusion, 8×8

print(f'\nComputing signed Grad-CAM for {len(chosen)} samples...')
t0 = time.time()
results = []
for c in chosen:
    H, W = c['mask'].shape
    pre  = compute_gradcam_signed(hybrid_model, c['x14'], c['s2_x'], layer_pre,  (H, W))
    fuse = compute_gradcam_signed(hybrid_model, c['x14'], c['s2_x'], layer_fuse, (H, W))
    joint = max(np.abs(pre).max(), np.abs(fuse).max(), 1e-8)
    pre_n, fuse_n = pre / joint, fuse / joint
    diff = norm_signed(fuse_n - pre_n)
    results.append({'mask': c['mask'], 'prob': c['prob'], 'pos_ratio': c['pos_ratio'],
                    'pre': pre_n, 'fuse': fuse_n, 'diff': diff})
print(f'Done in {time.time()-t0:.0f}s')

# Plot with a legend strip at the bottom
cam_cols = [('CNN bottleneck (pre-fusion, 8×8)', 'pre'),
            ('Fusion output (CNN+Clay, 8×8)',    'fuse'),
            ('Δ  (fusion − pre-fusion)',         'diff')]
n_rows = len(results)
ncols  = 3 + len(cam_cols)

# Extra row at the bottom for the legend strip
fig = plt.figure(figsize=(4.4*ncols, 4.2*n_rows + 1.2))
gs = fig.add_gridspec(n_rows + 1, ncols, height_ratios=[4.2]*n_rows + [0.6])
axes = np.array([[fig.add_subplot(gs[r, c]) for c in range(ncols)] for r in range(n_rows)])

for row, r in enumerate(results):
    mask, prob = r['mask'], r['prob']
    pred = (prob >= hybrid_best_thresh).astype(np.uint8)
    axes[row, 0].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f"GT (pos={r['pos_ratio']:.2%})")
    axes[row, 1].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'Pred (t={hybrid_best_thresh:.2f})')
    axes[row, 2].imshow(prob, cmap='RdYlGn', vmin=0, vmax=1)
    axes[row, 2].set_title('Probability')
    for j, (label, key) in enumerate(cam_cols):
        ax = axes[row, 3+j]
        ax.imshow(r[key], cmap='seismic', vmin=-1, vmax=1)
        ax.set_title(label, fontsize=9)
        ax.contour(mask, levels=[0.5], colors='black', linewidths=0.8)
    for j in range(ncols): axes[row, j].axis('off')

# Legend strip
# Color bar for signed Grad-CAM panels
cbar_ax = fig.add_subplot(gs[n_rows, 3:6])
gradient = np.linspace(-1, 1, 256).reshape(1, -1)
cbar_ax.imshow(gradient, aspect='auto', cmap='seismic', vmin=-1, vmax=1,
               extent=[-1, 1, 0, 1])
cbar_ax.set_yticks([])
cbar_ax.set_xticks([-1, 0, 1])
cbar_ax.set_xticklabels(['negative\ncontribution', 'neutral', 'positive\ncontribution'],
                        fontsize=9)
cbar_ax.set_title('Signed Grad-CAM (red increases output, blue decreases it). '
                  'Black contours: GT scar boundaries.',
                  fontsize=10, pad=8)

# Hide the unused legend-strip cells on the left
for c in list(range(0, 3)) + list(range(6, ncols)):
    ax = fig.add_subplot(gs[n_rows, c]); ax.axis('off')

fig.suptitle(f'{HYBRID_CFG["run_name"]} — Fusion Grad-CAM (signed) + Δ', fontsize=13, y=0.995)
plt.tight_layout(rect=[0, 0.02, 1, 0.985])
drive_dir = f'{HYBRID_CFG["save_dir"]}/{HYBRID_CFG["run_name"]}'
os.makedirs(drive_dir, exist_ok=True)
fig.savefig(f'{drive_dir}/gradcam_fusion_signed.png', dpi=150, bbox_inches='tight')
plt.show()

# 5. Test-set Δ aggregate: mean Δ inside vs outside GT scar
#    If fusion systematically shifts attention toward
#    or away from scars across the test set (not just the 5 chips).
print("\nComputing Δ aggregate over full test set...")
t0 = time.time()
inside_means, outside_means, pos_ratios = [], [], []
for c in candidates:
    if c['mask'].sum() == 0:           # skip empty-GT chips
        continue
    H, W = c['mask'].shape
    pre  = compute_gradcam_signed(hybrid_model, c['x14'], c['s2_x'], layer_pre,  (H, W))
    fuse = compute_gradcam_signed(hybrid_model, c['x14'], c['s2_x'], layer_fuse, (H, W))
    joint = max(np.abs(pre).max(), np.abs(fuse).max(), 1e-8)
    diff  = (fuse - pre) / joint
    mask_b = c['mask'] >= 0.5
    inside_means.append(float(diff[mask_b].mean()))
    outside_means.append(float(diff[~mask_b].mean()))
    pos_ratios.append(c['pos_ratio'])
print(f'Done in {time.time()-t0:.0f}s ({len(inside_means)} chips with non-empty GT)')

inside_means = np.array(inside_means)
outside_means = np.array(outside_means)
print(f"\nTest-set Δ aggregate:")
print(f"  mean Δ inside  scar contours: {inside_means.mean():+.4f}  (std {inside_means.std():.4f})")
print(f"  mean Δ outside scar contours: {outside_means.mean():+.4f}  (std {outside_means.std():.4f})")
print(f"  fraction of chips with Δ_inside > Δ_outside: "
      f"{(inside_means > outside_means).mean():.1%}")
print("\nInterpretation guide:")
print("  Δ_inside > Δ_outside  -> fusion systematically boosts attention on scars")
print("  Δ_inside < Δ_outside  -> fusion systematically shifts attention OFF scars")
print("  Δ_inside ≈ Δ_outside  -> fusion reorganizes but with no scar-vs-background bias")